In [1]:
# ===== 1: 설정 =====
import pymupdf          # PDF 파싱 및 텍스트/표 추출
import json             # 최종 레코드 저장
import re               # 패턴 매칭
from pathlib import Path
from collections import defaultdict

# 프로젝트 루트 (~/AI_programming/petmint-rag-chatbot)
PROJECT_DIR = Path.home() / "AI_programming" / "petmint-rag-chatbot"
RAW_PDF_DIR = PROJECT_DIR / "data" / "raw_pdf"   # 원본 PDF
CHUNKS_DIR  = PROJECT_DIR / "data" / "chunks"    # 최종 청크 JSON
DATA_DIR    = PROJECT_DIR / "data"               # limit_table.py 등 공유 산출물

PDF_FILES = {
    "강아지": RAW_PDF_DIR / "펫퍼민트_강아지.pdf",
    "고양이": RAW_PDF_DIR / "펫퍼민트_고양이.pdf",
}

for species, path in PDF_FILES.items():
    print(species, path.exists(), path)

강아지 True /home/kopo17/AI_programming/petmint-rag-chatbot/data/raw_pdf/펫퍼민트_강아지.pdf
고양이 True /home/kopo17/AI_programming/petmint-rag-chatbot/data/raw_pdf/펫퍼민트_고양이.pdf


In [2]:
# ===== 2: SECTION_MAP =====

SECTION_MAP = {
    "강아지": [
        {"start": 0,   "end": 0,   "section": "표지",          "section_name": None, "parser": None},  # 표지, 실질 정보 없어 스킵
        {"start": 1,   "end": 3,   "section": "안내_개인정보", "section_name": "개인신용정보 제공·이용에 대한 고객 권리 안내문", "parser": "parse_credit_info_notice"},
        {"start": 4,   "end": 7,   "section": "안내_개인정보", "section_name": "메리츠금융그룹 고객정보 취급방침", "parser": "parse_meritz_privacy_policy"},
        {"start": 8,   "end": 9,   "section": "안내_절차",    "section_name": "보험금 지급 절차", "parser": "parse_payment_procedure_notice"},
        {"start": 10,  "end": 10,  "section": "안내_절차",    "section_name": "보험품질보증제 및 민원상담 안내", "parser": "parse_misc_notice_p10" },
        {"start": 11,  "end": 11,  "section": "안내_절차",    "section_name": "보험 주요 민원 사례", "parser": None},  # OCR 미처리로 스킵 — 알려진 한계 참고
        {"start": 12, "end": 12,   "section": "약관_이용",    "section_name": "표지", "parser": None},  # 짧은 안내문구뿐, 실질 정보 없어 스킵       
        {"start": 13, "end": 16,   "section": "약관_이용",    "section_name": "기타문의처 안내", "parser": "parse_guide_contact_info"},
        {"start": 17,  "end": 30,  "section": "약관_이용",    "section_name": "약관 요약서", "parser": None},
        # 본문(보통약관/특별약관)과 내용 중복 + 도표·아이콘 위주라 파싱 가치 낮음, 스킵
        # (한도 숫자는 limit_table.py로 이미 별도 확정돼 있음, 06번 품질검증용)
        {"start": 31,  "end": 33,  "section": "목차",         "section_name": None, "parser": None},
        {"start": 34,  "end": 34,  "section": "간지",         "section_name": "표지", "parser": None},
        {"start": 35,  "end": 37,  "section": "안내_요약",    "section_name": "가입자 유의사항", "parser": "parse_subscriber_notice"},
        {"start": 38,  "end": 45,  "section": "안내_요약",    "section_name": "주요내용 요약서", "parser": "parse_summary_notice"},
        {"start": 46,  "end": 48,  "section": "안내_용어",    "section_name": "보험용어 해설", "parser": "parse_terms_glossary"},
        {"start": 49, "end": 50,   "section": "간지",         "section_name": None, "parser": None},  # 보통약관 간지 + 빈 표지
        {"start": 51,  "end": 86,  "section": "보통약관",     "section_name": None, "parser": "parse_articles"},
        {"start": 87,  "end": 88,  "section": "간지",         "section_name": None, "parser": None},
        {"start": 89,  "end": 107, "section": "특별약관",     "section_name": "반려동물 비용손해 관련 특별약관 일반조항", "parser": "parse_articles"},
        {"start": 108, "end": 117, "section": "특별약관",     "section_name": "펫퍼민트 반려견 통원의료비보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 118, "end": 127, "section": "특별약관",     "section_name": "펫퍼민트 반려견 입원의료비보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 128, "end": 137, "section": "특별약관",     "section_name": "펫퍼민트 반려견 통원의료비Ⅱ보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 138, "end": 147, "section": "특별약관",     "section_name": "펫퍼민트 반려견 입원의료비Ⅱ보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 148, "end": 160, "section": "특별약관",     "section_name": "펫퍼민트 반려견 통원의료비Ⅲ보장 특별약관", "parser": "parse_medical_teukyak_3stage"},
        {"start": 161, "end": 173, "section": "특별약관",     "section_name": "펫퍼민트 반려견 입원의료비Ⅲ보장 특별약관", "parser": "parse_medical_teukyak_3stage"},
        {"start": 174, "end": 180, "section": "특별약관",     "section_name": "민사소송법률비용Ⅱ보장 특별약관", "parser": "parse_articles"},
        {"start": 181, "end": 192, "section": "특별약관",     "section_name": "배상책임 관련 특별약관 일반조항", "parser": "parse_articles"},
        {"start": 193, "end": 195, "section": "특별약관",     "section_name": "갱신형 펫퍼민트 반려견 배상책임보장 특별약관", "parser": "parse_articles"},
        {"start": 196, "end": 198, "section": "특별약관",     "section_name": "자동갱신 특별약관", "parser": "parse_articles"},
        {"start": 199, "end": 201, "section": "특별약관",     "section_name": "반려동물 특정 질병 보장제한부 인수 특별약관", "parser": "parse_articles"},
        {"start": 202, "end": 205, "section": "별첨",         "section_name": "특정질병 분류표(반려견)", "parser": "parse_teukjeong_table"},
        {"start": 206, "end": 207, "section": "별표",         "section_name": "【별표1】 보험금을 지급할 때의 적립이율 계산", "parser": "parse_table_only_section"},
        {"start": 208, "end": 238, "section": "별표",         "section_name": "【별표2】 장해분류표", "parser": "parse_byeopyo2"},
        {"start": 239, "end": 239, "section": "별표",         "section_name": "【별표3】 소송목적의 값에 따른 변호사비용", "parser": "parse_table_only_section"},
        {"start": 240, "end": 240, "section": "별표",         "section_name": "【별표4】 민사소송 인지액 /【별표5】 송달료", "parser": "parse_table_only_section"},
        {"start": 241, "end": 244, "section": "법규",         "section_name": "【법규1】 개인정보 보호법", "parser": "parse_bupgyu"},
        {"start": 245, "end": 253, "section": "법규",         "section_name": "【법규2】 금융소비자 보호에 관한 법률", "parser": "parse_bupgyu"},
        {"start": 254, "end": 254, "section": "법규",         "section_name": "【법규3】 민법", "parser": "parse_bupgyu"},
        {"start": 255, "end": 260, "section": "법규",         "section_name": "【법규4】 신용정보의 이용 및 보호에 관한 법률", "parser": "parse_bupgyu"},
        {"start": 261, "end": 261, "section": "법규",         "section_name": "【법규5】 약사법", "parser": "parse_bupgyu"},
        {"start": 262, "end": 263, "section": "법규",         "section_name": "【법규6】 민사소송법", "parser": "parse_bupgyu"},
        {"start": 264, "end": 266, "section": "법규",         "section_name": "【법규7】 소비자기본법", "parser": "parse_bupgyu"},
        {"start": 267, "end": 268, "section": "기타_제도성_특별약관",     "section_name": "펫퍼민트 기부 특별약관", "parser": "parse_articles"},
        {"start": 269, "end": 269, "section": "기타_제도성_특별약관",     "section_name": "보험료 자동납입 특별약관", "parser": "parse_articles"},
        {"start": 270, "end": 271, "section": "기타_제도성_특별약관",     "section_name": "전자서명 특별약관", "parser": "parse_articles"},
        {"start": 272, "end": 276, "section": "기타_제도성_특별약관",     "section_name": "장애인전용보험전환 특별약관", "parser": "parse_articles"},
        {"start": 277, "end": 280, "section": "기타_제도성_특별약관",     "section_name": "이륜자동차 운전중 상해 부보장 특별약관", "parser": "parse_articles"},
        {"start": 281, "end": 284, "section": "기타_제도성_특별약관",     "section_name": "지정대리청구서비스 특별약관", "parser": "parse_articles"},
        {"start": 285, "end": 289, "section": "기타_제도성_특별약관",     "section_name": "보장성보험 보험료 납입유예 특별약관", "parser": "parse_articles"},
        {"start": 290, "end": 291, "section": "색인",         "section_name": None, "parser": None},
    ],
    "고양이": [
        {"start": 0, "end": 0, "section": "표지", "section_name": None, "parser": None},  # 표지, 실질 정보 없어 스킵
        {"start": 1, "end": 3, "section": "안내_개인정보", "section_name": "개인신용정보 제공·이용에 대한 고객 권리 안내문", "parser": "parse_credit_info_notice"},
        {"start": 4, "end": 7, "section": "안내_개인정보", "section_name": "메리츠금융그룹 고객정보 취급방침", "parser": "parse_meritz_privacy_policy"},
        {"start": 8, "end": 9, "section": "안내_절차", "section_name": "보험금 지급 절차", "parser": "parse_payment_procedure_notice"},
        {"start": 10, "end": 10, "section": "안내_절차", "section_name": "보험품질보증제 및 민원상담 안내", "parser": "parse_misc_notice_p10"},
        {"start": 11, "end": 11, "section": "안내_절차", "section_name": "보험 주요 민원 사례", "parser": None},  # 이미지 페이지(OCR 미처리)로 스킵 — 강아지 p11과 동일
        {"start": 12, "end": 12, "section": "약관_이용", "section_name": "표지", "parser": None},  # 짧은 안내문구뿐, 실질 정보 없어 스킵
        {"start": 13, "end": 16, "section": "약관_이용", "section_name": "기타문의처 안내", "parser": "parse_guide_contact_info"},
        {"start": 17, "end": 29, "section": "약관_이용", "section_name": "약관 요약서", "parser": None},  # 본문과 중복 + 도표 위주라 스킵(한도는 limit_table.py로 별도 확정)
        {"start": 30, "end": 32, "section": "목차", "section_name": None, "parser": None},
        {"start": 33, "end": 33, "section": "간지", "section_name": None, "parser": None},  # 보통약관 간지
        {"start": 34, "end": 35, "section": "안내_요약", "section_name": "가입자 유의사항", "parser": "parse_subscriber_notice"},
        {"start": 36, "end": 42, "section": "안내_요약", "section_name": "주요내용 요약서", "parser": "parse_summary_notice"},
        {"start": 43, "end": 44, "section": "안내_용어", "section_name": "보험용어 해설", "parser": "parse_terms_glossary"},
        {"start": 45, "end": 46, "section": "간지", "section_name": None, "parser": None},  # 간지 + 빈 표지
        {"start": 47, "end": 82, "section": "보통약관", "section_name": None, "parser": "parse_articles"},
        {"start": 83, "end": 84, "section": "간지", "section_name": None, "parser": None},  # 간지 + 빈 표지
        {"start": 85, "end": 103, "section": "특별약관", "section_name": "반려동물 비용손해 관련 특별약관 일반조항", "parser": "parse_articles"},
        {"start": 104, "end": 112, "section": "특별약관", "section_name": "펫퍼민트 반려묘 통원의료비보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 113, "end": 122, "section": "특별약관", "section_name": "펫퍼민트 반려묘 입원의료비보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 123, "end": 131, "section": "특별약관", "section_name": "펫퍼민트 반려묘 통원의료비Ⅱ보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 132, "end": 140, "section": "특별약관", "section_name": "펫퍼민트 반려묘 입원의료비Ⅱ보장 특별약관", "parser": "parse_medical_teukyak_2stage"},
        {"start": 141, "end": 152, "section": "특별약관", "section_name": "펫퍼민트 반려묘 통원의료비Ⅲ보장 특별약관", "parser": "parse_medical_teukyak_3stage"},
        {"start": 153, "end": 164, "section": "특별약관", "section_name": "펫퍼민트 반려묘 입원의료비Ⅲ보장 특별약관", "parser": "parse_medical_teukyak_3stage"},
        {"start": 165, "end": 171, "section": "특별약관", "section_name": "민사소송법률비용Ⅱ보장 특별약관", "parser": "parse_articles"},
        {"start": 172, "end": 174, "section": "특별약관", "section_name": "반려동물 특정 질병 보장제한부 인수 특별약관", "parser": "parse_articles"},
        {"start": 175, "end": 179, "section": "별첨", "section_name": "특정질병 분류표(반려묘)", "parser": "parse_teukjeong_table"},
        {"start": 180, "end": 181, "section": "별표", "section_name": "【별표1】 보험금을 지급할 때의 적립이율 계산", "parser": "parse_table_only_section"},
        {"start": 182, "end": 212, "section": "별표", "section_name": "【별표2】 장해분류표", "parser": "parse_byeopyo2"},
        {"start": 213, "end": 213, "section": "별표", "section_name": "【별표3】 소송목적의 값에 따른 변호사비용", "parser": "parse_table_only_section"},
        {"start": 214, "end": 214, "section": "별표", "section_name": "【별표4】 민사소송 인지액 /【별표5】 송달료", "parser": "parse_table_only_section"},
        {"start": 215, "end": 218, "section": "법규", "section_name": "【법규1】 개인정보 보호법", "parser": "parse_bupgyu"},
        {"start": 219, "end": 227, "section": "법규", "section_name": "【법규2】 금융소비자 보호에 관한 법률", "parser": "parse_bupgyu"},
        {"start": 228, "end": 228, "section": "법규", "section_name": "【법규3】 민법", "parser": "parse_bupgyu"},
        {"start": 229, "end": 234, "section": "법규", "section_name": "【법규4】 신용정보의 이용 및 보호에 관한 법률", "parser": "parse_bupgyu"},
        {"start": 235, "end": 235, "section": "법규", "section_name": "【법규5】 약사법", "parser": "parse_bupgyu"},
        {"start": 236, "end": 237, "section": "법규", "section_name": "【법규6】 민사소송법", "parser": "parse_bupgyu"},
        {"start": 238, "end": 238, "section": "법규", "section_name": "【법규7】 소비자기본법", "parser": "parse_bupgyu"},
        {"start": 239, "end": 240, "section": "간지", "section_name": None, "parser": None},  # 기타 제도성 특별약관 간지 + 빈 표지
        {"start": 241, "end": 242, "section": "기타_제도성_특별약관", "section_name": "펫퍼민트 Cat 기부 특별약관", "parser": "parse_articles"},
        {"start": 243, "end": 243, "section": "기타_제도성_특별약관", "section_name": "보험료 자동납입 특별약관", "parser": "parse_articles"},
        {"start": 244, "end": 245, "section": "기타_제도성_특별약관", "section_name": "전자서명 특별약관", "parser": "parse_articles"},
        {"start": 246, "end": 250, "section": "기타_제도성_특별약관", "section_name": "장애인전용보험전환 특별약관", "parser": "parse_articles"},
        {"start": 251, "end": 254, "section": "기타_제도성_특별약관", "section_name": "이륜자동차 운전중 상해 부보장 특별약관", "parser": "parse_articles"},
        {"start": 255, "end": 258, "section": "기타_제도성_특별약관", "section_name": "지정대리청구서비스 특별약관", "parser": "parse_articles"},
        {"start": 259, "end": 263, "section": "기타_제도성_특별약관", "section_name": "보장성보험 보험료 납입유예 특별약관", "parser": "parse_articles"},
        {"start": 264, "end": 265, "section": "색인", "section_name": None, "parser": None},
    ],
}

def get_section(species: str, page: int):
    for entry in SECTION_MAP[species]:
        if entry["start"] <= page <= entry["end"]:
            return entry["section"], entry["section_name"]
    return None, None

# 검증: 전체 페이지가 빠짐없이 매핑되는지 확인
for species, path in PDF_FILES.items():
    doc_check = pymupdf.open(path)
    n = len(doc_check)
    missing = [p for p in range(n) if get_section(species, p) == (None, None)]
    print(f"{species}: 전체 {n}쪽, 매핑 안 된 쪽: {missing}")
    doc_check.close()

강아지: 전체 292쪽, 매핑 안 된 쪽: []
고양이: 전체 266쪽, 매핑 안 된 쪽: []


In [3]:
# ===== 3: 텍스트 파싱 범용 도구 =====
# 여러 섹션 전용 함수가 공유해서 쓰는 저수준 패턴 분리 도구 모음
import re

# 숫자목록("1. 제목") 분리 — parse_credit_info_notice, parse_meritz_privacy_policy(I.)에서 사용
NUMBERED_TITLE = re.compile(r"\n+\s*(\d{1,2})\.[ \t]+([^\n]+)(?=\n)")
def split_by_numbered_title(text: str):
    matches = list(NUMBERED_TITLE.finditer(text))
    if not matches:
        return [{"num": None, "title": None, "text": text}]
    segments = []
    if matches[0].start() > 0:
        segments.append({"num": None, "title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"num": int(m.group(1)), "title": m.group(2).strip(), "text": text[start:end]})
    return segments

# 가나다("가. 소제목") 분리, 순서 역행하는 가짜 매치 거름 — parse_credit_info_notice, parse_meritz_privacy_policy(I.)에서 사용
KOREAN_SUB_TITLE = re.compile(r"\n+\s*(?:\d{1,3}\s*)?\s*(가|나|다|라|마|바|사|아|자|차|카|타|파|하)\.[ \t]*([^\n]+)(?=\n)")
KOREAN_ORDER = "가나다라마바사아자차카타파하"

def _is_wrapped_word_end(text: str, m) -> bool:
    """'…치료 후 평가한\n다. 다만, …'처럼 줄바꿈으로 끊긴 단어 끝 '다.'를 목 라벨로 오인했는지 판별.
    빈 줄·페이지 번호 없이 바로 앞 줄에 붙어 있고, 그 줄이 한글로 끝나며 제목줄(1. / 가.)이 아니면 단어 이어짐이다.
    (별표2 신경계ㆍ정신행동 장해에서 sub_title이 '다만, 진단시점에…'로 오파싱된 건 — 양 종 각 1건, 다른 오탐 0)"""
    gap = text[m.start():m.start(1)]
    if gap.count("\n") != 1 or re.search(r"\d", gap):
        return False
    prev = text[:m.start()].rstrip("\n").split("\n")[-1]
    return bool(re.search(r"[가-힣]$", prev.rstrip())) and not re.match(r"^\s*(\d{1,2}|[가-하])\.\s", prev)

def split_by_korean_sub(text: str):
    """'가. 소제목' 패턴으로 나눈다. 순서가 역행하는(가짜) 라벨과 줄바꿈으로 끊긴 단어 끝('한\n다.')은
    무시하고 이전 항목에 이어붙인다.
    조각은 라벨 글자부터 시작한다 — 라벨 앞에 끼어든 페이지 번호('209\n나. 장해판정기준')는 어느 조각에도 넣지 않는다."""
    matches = [m for m in KOREAN_SUB_TITLE.finditer(text) if not _is_wrapped_word_end(text, m)]
    if not matches:
        return [{"label": None, "title": None, "text": text}]

    valid_matches = []
    last_order_idx = -1
    for m in matches:
        label = m.group(1)
        order_idx = KOREAN_ORDER.index(label)
        if order_idx > last_order_idx:
            valid_matches.append(m)
            last_order_idx = order_idx

    matches = valid_matches
    if not matches:
        return [{"label": None, "title": None, "text": text}]

    segments = []
    if matches[0].start() > 0:
        segments.append({"label": None, "title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start(1)
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"label": m.group(1), "title": m.group(2).strip(), "text": text[start:end]})
    return segments

# 숫자→가나다 2단계 조합 (가나다 레벨이 청크) — parse_credit_info_notice, parse_meritz_privacy_policy에서 사용
def split_numbered_then_korean(text: str) -> list[dict]:
    """'1. 제목' → 그 안에서 '가. 소제목'으로 2단계 분리 (numbered_then_korean 전략)"""
    results = []
    numbered_segments = split_by_numbered_title(text)
    for seg in numbered_segments:
        sub_segments = split_by_korean_sub(seg["text"])
        for sub in sub_segments:
            results.append({
                "item_num": seg["num"],
                "item_title": seg["title"],
                "sub_label": sub["label"],
                "sub_title": sub["title"],
                "text": sub["text"],
            })
    return results
    
# 로마자("I./II./III.") 분리 — parse_meritz_privacy_policy 전용
# (유형구분용 로마숫자 Ⅰ·Ⅱ·Ⅲ와는 다른 영어 알파벳 I이니 혼동 주의)
ROMAN_ALPHA_TITLE = re.compile(r"\n+\s*(I{1,3}V?|IV|V)\.\s*([^\n]+)(?=\n)")
def split_by_roman_alpha_title(text: str):
    """'I. 제목' 패턴(영어 알파벳 I/II/III)으로 텍스트를 나눈다."""
    matches = list(ROMAN_ALPHA_TITLE.finditer(text))
    if not matches:
        return [{"roman": None, "title": None, "text": text}]
    segments = []
    if matches[0].start() > 0:
        segments.append({"roman": None, "title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"roman": m.group(1), "title": m.group(2).strip(), "text": text[start:end]})
    return segments

# 원문자("① 제목") 분리 — parse_meritz_privacy_policy(표, III.)에서 사용
CIRCLE_NUMBER = re.compile(r"\n+\s*([①②③④⑤⑥⑦⑧⑨⑩])\s*([^\n]+)(?=\n)")

def split_by_circle_number(text: str):
    """'① 제목' 패턴(원문자)으로 텍스트를 나눈다."""
    matches = list(CIRCLE_NUMBER.finditer(text))
    if not matches:
        return [{"circle": None, "title": None, "text": text}]
    segments = []
    if matches[0].start() > 0:
        segments.append({"circle": None, "title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"circle": m.group(1), "title": m.group(2).strip(), "text": text[start:end]})
    return segments

# 네모("■ 제목") 분리, 페이지번호가 줄바꿈 없이 붙는 경우도 허용 — parse_payment_procedure_notice 전용
SQUARE_TITLE = re.compile(r"(?:\n+\s*|\d{1,3}\s*)■\s*([^\n]+)(?=\n)")

def split_by_square_title(text: str):
    """'■ 제목' 패턴으로 텍스트를 나눈다."""
    matches = list(SQUARE_TITLE.finditer(text))
    if not matches:
        return [{"title": None, "text": text}]
    segments = []
    if matches[0].start() > 0:
        segments.append({"title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start() + m.group(0).index("■")   # 기호 앞에 붙은 페이지 번호('8\n■ 의료심사')는 버린다
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"title": m.group(1).strip(), "text": text[start:end]})
    return segments
    
SQUARE_BOX_TITLE = re.compile(r"(?:\n+\s*|\d{1,3}\s*)□\s*([^\n]+)(?=\n)")
CIRCLE_BULLET = re.compile(r"\n+\s*○\s*([^\n]+)(?=\n)")

# 네모박스("□ 제목") 분리, 페이지번호가 줄바꿈 없이 붙는 경우도 허용 — parse_subscriber_notice 전용
def split_by_square_box(text: str):
    """'□ 제목' 패턴으로 텍스트를 나눈다."""
    matches = list(SQUARE_BOX_TITLE.finditer(text))
    if not matches:
        return [{"title": None, "text": text}]
    segments = []
    if matches[0].start() > 0:
        segments.append({"title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start() + m.group(0).index("□")   # 기호 앞에 붙은 페이지 번호('8\n■ 의료심사')는 버린다
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"title": m.group(1).strip(), "text": text[start:end]})
    return segments

# 동그라미 불릿("○ 제목") 분리 — parse_subscriber_notice 전용
def split_by_circle_bullet(text: str):
    """'○ 제목' 패턴으로 텍스트를 나눈다."""
    matches = list(CIRCLE_BULLET.finditer(text))
    if not matches:
        return [{"title": None, "text": text}]
    segments = []
    if matches[0].start() > 0:
        segments.append({"title": None, "text": text[:matches[0].start()]})
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        segments.append({"title": m.group(1).strip(), "text": text[start:end]})
    return segments
    
# 등급(고급형/기본형/실속형) 분리 — parse_articles에서 사용, 통원/입원의료비 특약에만 등급 구분이 있다
TYPE_PATTERN = re.compile(r"[①②③]\s*(고급형|기본형|실속형)")
# 등급 라벨만 남은 빈 조각 판별용 — "① 고급형" 직후 바로 \uf000가 와서 생긴다
COVERAGE_LABEL_ONLY = re.compile(r"^\s*[①②③]\s*(고급형|기본형|실속형)\s*$")

def split_by_coverage_type(article_text: str):
    """'① 고급형' 패턴으로 조 본문을 등급별로 나눈다. 등급 구분이 없으면 통째로 하나."""
    matches = list(TYPE_PATTERN.finditer(article_text))
    if not matches:
        return [{"coverage_type": None, "text": article_text}]

    segments = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(article_text)
        segments.append({"coverage_type": m.group(1), "text": article_text[start:end]})
    if matches[0].start() > 0:
        prefix = article_text[:matches[0].start()]
        segments.insert(0, {"coverage_type": None, "text": prefix})
    return segments

# 법규 섹션의 항 분리용 — 약관은 \uf000를 쓰지만 법규는 실제 원문자(①②③)를 쓴다
CIRCLE_NUMS = "①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮"
CIRCLE_TO_NUM = {c: i for i, c in enumerate(CIRCLE_NUMS, start=1)}
PARAGRAPH_CIRCLE_SPLIT = re.compile(rf"(?=[{CIRCLE_NUMS}])")

# 내용 없이 "삭제"/"(생략)"만 있는 항 — 법 개정으로 삭제됐거나 발췌본에서 생략된 자리라 청크로 둘 가치가 없다
EMPTY_PARAGRAPH = re.compile(r"^\s*[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮]?\s*\(?\s*(삭제|생략)\s*\)?\s*$")

ARTICLE_HEADER = re.compile(r"제(\d{1,3})조\s*\(([^)]{1,60})\)")

def find_article_headers(sorted_text: str):
    """sort=True로 뽑은 텍스트에서 조 헤더 목록을 (조번호, 제목, 매치위치)로 반환."""
    return [(int(m.group(1)), m.group(2), m.start()) for m in ARTICLE_HEADER.finditer(sorted_text)]

# 조 본문에 【별첨】이 나오면 거기서 끊는다 — 별첨은 별도 파서가 처리하므로 겹치지 않게
ATTACHMENT_MARKER = re.compile(r"【\s*별첨\s*】")

# 약관의 항 마커 (법규는 실제 원문자 ①②③를 쓴다)
PARAGRAPH_MARKER = "\uf000"

def split_into_paragraphs(article_text: str):
    """한 조의 본문에서 \uf000 등장 순서대로 1항, 2항...으로 나눈다."""
    parts = article_text.split(PARAGRAPH_MARKER)
    if len(parts) == 1:
        return [{"num": None, "text": article_text.strip()}]
    result = [{"num": None, "text": parts[0].strip()}] if parts[0].strip() else []
    for i, p in enumerate(parts[1:], start=1):
        result.append({"num": i, "text": p.strip()})
    return result

In [4]:
# ===== 4: 표(테이블 파싱) 범용 도구 =====
# ===== 표 추출 + 본문 위치 매칭 =====
def make_table_probe(rows) -> str:
    """표에서 본문 위치를 찾기 위한 탐침 문자열을 고른다.
    셀 텍스트 전체는 본문에서 용어가 중간에 끼어들어 연속 매칭이 안 되므로 한 줄 단위로 쓴다.
    첫 줄이 짧은 표(세로로 쪼개진 표)도 있어 모든 줄을 훑고, 8자 이상인 첫 줄을 쓴다."""
    for r in rows:
        for cell in r:
            if not cell:
                continue
            for line in cell.split("\n"):
                line = line.strip()
                if len(line) >= 8:
                    return line
    return ""

def extract_tables_with_offset(pages: list[dict], pdf_path, verbose: bool = True) -> list[tuple[int, str]]:
    """페이지 리스트에서 표를 꺼내, 각 표가 본문(이어붙인 full_text)의 어느 위치에 있는지와 함께 반환한다.
    같은 페이지에 머리글이 같은 표가 여러 개 있을 수 있으므로, 탐색 커서를 전진시켜
    각 표가 서로 다른 위치에 매칭되게 한다(표와 본문 모두 위→아래 순서라 가능).
    탐침을 본문에서 못 찾는 표는 버리고 로그로 남긴다 — 본문 문단을 표로 오인식한 가짜 표가 주로 걸린다.
    반환: [(full_text상 위치, 표 markdown), ...] 위치 오름차순"""
    pdf = pymupdf.open(pdf_path)
    result = []
    base = 0
    for p in pages:
        body = p["text"]
        cursor = 0                      # ← 같은 페이지 안에서 전진하는 탐색 커서
        tabs = pdf[p["page"]].find_tables()
        for t in tabs.tables:
            rows = t.extract()
            probe = make_table_probe(rows)
            if not probe:
                if verbose:
                    print(f"  [표 스킵] {p['page']}쪽: 탐침 없음(셀이 너무 짧음)")
                continue
            local_pos = body.find(probe, cursor)
            if local_pos == -1:
                if verbose:
                    print(f"  [표 스킵] {p['page']}쪽: 본문에서 못 찾음 — {probe[:30]}")
                continue
            cursor = local_pos + len(probe)      # 다음 표는 이 뒤에서 찾는다
            md = trim_to_real_table(clean_fake_columns(t.to_markdown()))
            if md.strip():
                result.append((base + local_pos, md))
        base += len(body) + 1
    pdf.close()
    result.sort(key=lambda x: x[0])
    return result


# ===== 청크에 표 붙이기 (글자 위치 기준) =====
def attach_tables_by_offset(chunks: list[dict], tables: list[tuple[int, str]]) -> list[dict]:
    """각 표를 글자 위치가 속한 청크에 붙인다.
    표마다 위치가 하나씩이라 한 청크에만 붙으며, 기존의 페이지 단위 중복 방지가 필요 없다.
    청크는 start_offset/end_offset을 갖고 있어야 한다."""
    for c in chunks:
        mine = [md for pos, md in tables if c["start_offset"] <= pos < c["end_offset"]]
        c["content"] = c["text"] + ("\n\n[표]\n" + "\n\n".join(mine) if mine else "")
    return chunks

# ===== 통원Ⅰ·Ⅱ·Ⅲ, 입원Ⅰ·Ⅱ·Ⅲ 한도표 재구성 (pymupdf 표 레이아웃 파싱 오류 수정) =====
def rebuild_level2_table(content: str) -> str:
    """통원/입원의료비 Ⅰ·Ⅱ 보장 특약의 깨진 한도표를 읽을 수 있는 텍스트 표로 교체한다.
    pymupdf가 이 표를 제대로 추출하지 못한다 — 세로쓰기 라벨('통원/의료비')과
    병합된 자기부담금 셀이 행 사이에 끼어들어 순서가 뒤엉킨다.
    본문에서 '1일당 N만원' 한도와 'X만원/Y만원' 자기부담금만 정규식으로 뽑아 재조립한다.
    자기부담금은 원문에서 두 행에 걸쳐 병합된 하나의 값이므로 두 행에 동일하게 넣는다(좌표로 확인함)."""
    start_match = re.search(r"항목\s*자기부담금\s*지급 한도", content)
    end_idx = content.find("【보험금 지급금액 산출방식】")
    if not start_match or end_idx == -1:
        return content
    start_idx = start_match.start()

    broken_segment = content[start_idx:end_idx]
    limits = re.findall(r"1일당\s*(\d+)만원", broken_segment)
    copay_match = re.search(r"(\d+)만원/(\d+)만원", broken_segment)

    if len(limits) < 2 or not copay_match:
        print("  !! Lv2 파싱 실패:", "limits=", limits, "copay=", copay_match)
        return content

    no_surgery, surgery = limits[:2]
    copay1, copay2 = copay_match.groups()
    verb = "통원" if "통원 중" in broken_segment else "입원"
    copay_text = f"1일당 {copay1}만원/{copay2}만원 중 보험증권에 기재된 자기부담금"

    clean_table = f"""
      항목                                  자기부담금                                      지급 한도
      {verb} 중 수술을 하지 않은 날의 경우    {copay_text}    1일당 {no_surgery}만원
      {verb} 중 수술을 한 날의 경우          {copay_text}    수술당일에 한하여 1일당 {surgery}만원

"""
    return content[:start_idx] + clean_table + content[end_idx:]

def rebuild_level3_table(content: str) -> str:
    """통원/입원의료비 Ⅲ 보장 특약의 깨진 한도표를 읽을 수 있는 텍스트 표로 교체한다.
    Ⅲ유형은 MRI·CT·내시경처치 여부와 연간 횟수까지 조건이 갈려 4단계 구조이며,
    세로쓰기 라벨('통원/의료비/Ⅲ')과 병합된 자기부담금 셀이 글자 단위로 흩어져 행 순서가 완전히 뒤엉킨다.
    본문에서 '1일당 N만원' 한도 4개와 'X만원/Y만원' 자기부담금만 정규식으로 뽑아 재조립한다.
    자기부담금은 원문에서 네 행에 걸쳐 병합된 하나의 값이므로 모든 행에 동일하게 넣는다."""
    start_match = re.search(r"자기\s*\n\s*항목", content)
    end_idx = content.find("【보험금 지급금액 산출방식】")
    if not start_match or end_idx == -1:
        return content
    start_idx = start_match.start()

    broken_segment = content[start_idx:end_idx]
    limits = re.findall(r"1일당\s*(\d+)만원", broken_segment)
    copay1_match = re.search(r"(\d+)만원/", broken_segment)
    copay2_match = re.search(r"(\d+)만원원", broken_segment)

    if len(limits) < 4 or not copay1_match or not copay2_match:
        print("  !! Lv3 파싱 실패:", "limits=", limits, "copay1=", copay1_match, "copay2=", copay2_match)
        return content

    first_mri, second_mri, no_mri, surgery = limits[:4]
    copay1, copay2 = copay1_match.group(1), copay2_match.group(1)
    verb = "통원" if "통원 중" in broken_segment else "입원"
    copay_text = f"1일당 {copay1}만원/{copay2}만원 중 보험증권에 기재된 자기부담금"

    clean_table = f"""
      항목                                                        자기부담금                                      지급 한도
      {verb} 중 수술을 하지 않은 날 - MRI,CT 및 내시경처치를 받은 날, 연간 첫 번째    {copay_text}    1일당 {first_mri}만원
      {verb} 중 수술을 하지 않은 날 - MRI,CT 및 내시경처치를 받은 날, 연간 두 번째 이상  {copay_text}    1일당 {second_mri}만원
      {verb} 중 수술을 하지 않은 날 - MRI,CT 및 내시경처치를 받지 않은 날              {copay_text}    1일당 {no_mri}만원
      {verb} 중 수술을 한 날의 경우                                        {copay_text}    수술당일에 한하여 1일당 {surgery}만원

"""
    return content[:start_idx] + clean_table + content[end_idx:]

In [5]:
# ===== 5: citation 범용 도구 =====

CITE_PATTERN = re.compile(r"제\d+조(?:제\d+항)?(?:제\d+호)?|제\d+항|제\d+호")

# ===== citations 추출 — 조+항/호, 조+호, 단독조, 단독항/호, 단독호 전부 포착 =====

CITE_PATTERN_FULL = re.compile(
    r"제(\d+)조(?:\([^)]{0,40}\))?\s*제\s*(\d+)항(?:\s*제\s*(\d+)호)?"   # 조+항(+호)
    r"|제(\d+)조(?:\([^)]{0,40}\))?\s*제\s*(\d+)호"                      # 조+호 (항 없이)
    r"|제(\d+)조"                                                          # 단독 조
    r"|제\s*(\d+)항(?:\s*제\s*(\d+)호)?(?!\s*의\s*\d)"                     # 단독 항(+호)
    r"|제\s*(\d+)호"                                                       # 단독 호
)

RANGE_PATTERN = re.compile(r"제\s*(\d+)\s*(항|호)\s*(?:부터|내지)\s*제\s*(\d+)\s*(항|호)")


def extract_law_name(preceding: str) -> str | None:
    """citation 바로 앞 텍스트에서 법률명을 찾는다.
    낫표(「」『』), 둥근괄호, 약관의 inline 법률 박스에 쓰이는 【】 모두 처리한다.
    '반소(민사소송법 제269조'처럼 여는 괄호가 법명 바로 앞에 붙는 경우도 고려해
    마지막 토큰에서 괄호류 이후 부분만 떼어내 비교한다.
    위 원본 로직이 못 찾을 때만 추가 폴백으로 ① 시행령/시행규칙이 붙은 법명,
    ② 긴 법명이 줄바꿈으로 쪼개진 경우(법률/시행령이 두 줄로 나뉨)를 보정해 잡는다."""
    bracket_match = re.search(r"[「『｢(【]([^「『｢(【」』｣)】]{1,40}법(?:률)?)[」』｣)】]\s*$", preceding)
    if bracket_match:
        return bracket_match.group(1).strip()
    tokens = preceding.strip().split()
    if tokens:
        last_token = re.split(r"[(\[{「『｢【]", tokens[-1])[-1]
        if re.fullmatch(r"[가-힣·]{1,30}법(?:률)?", last_token):
            return last_token
    # ===== 추가 폴백 (위에서 못 찾은 경우에만) =====
    # 법명이 줄바꿈으로 쪼개진 것 보정: '법 률'→'법률', '시\n행령/규칙'→'시행령/규칙'
    s = re.sub(r"법\s+률", "법률", preceding)
    s = re.sub(r"시\s*\n\s*행", "시행", s)
    s = re.sub(r"시행\s*규\s+칙", "시행규칙", s)          # '｢산업재해보상보험법 시행규\n칙｣'
    s = re.sub(r"(?<=\s)\d{1,4}(?=\s)", " ", s)          # 법명 사이에 낀 페이지 번호('동 199 법 시행령')
    # (a) 닫는 괄호로 끝나는 「법명(+시행령/규칙)」 — 긴 법명·법률 줄바꿈, 시행령·시행규칙 포함
    m = re.search(r"[「『｢(【]([^「『｢(【」』｣)】]{1,60}법(?:률)?(?:\s*시행(?:령|규칙))?)[」』｣)】]\s*$", s)
    if m:
        return m.group(1).strip()
    # (b) 여는 괄호 뒤 법명(+시행령/규칙)이 citation 바로 앞까지 (닫는 괄호는 조번호 뒤)
    m = re.search(r"[「『｢(【]\s*([가-힣][가-힣\s]{0,20}법(?:률)?(?:\s*시행(?:령|규칙))?)\s*$", s)
    if m:
        return m.group(1).strip()
    # (c) 괄호 없는 '상법 시행령 제44조의2' — 마지막 토큰이 시행령/시행규칙이고 그 앞 토큰이 법명
    m = re.search(r"(?<![가-힣])((?:동\s*)?[가-힣·]{0,30}법(?:률)?)\s+(시행(?:령|규칙))\s*$", s)
    if m:
        return f"{re.sub(chr(32), '', m.group(1)) if m.group(1).startswith('동') else m.group(1)} {m.group(2)}"
    return None


def clean_law_name(law_name: str) -> str:
    """저장용: 줄바꿈 등 불필요한 공백 정리, '법'/'법률' 바로 앞 공백만 제거."""
    cleaned = re.sub(r"\s+", " ", law_name).strip()
    cleaned = re.sub(r"\s+(법률?)$", r"\1", cleaned)
    return cleaned


def expand_range_citations(text: str, emit_from: int = 0, spans: dict = None) -> list[dict]:
    """'제1항부터 제3항까지' 같은 범위 표현을 1,2,3으로 모두 펼친다. emit_from 이전 위치의 범위는 내보내지 않는다.
    펼친 항목은 범위 첫 항목(spans[시작 위치])이 해석된 조·항·법명·문서명을 그대로 잇는다
    — '제3항 제3호 내지 제4호'의 3·4호는 3항 안의 호다(호 번호를 항으로 두지 않는다)."""
    spans = spans or {}
    results = []
    for m in RANGE_PATTERN.finditer(text):
        if m.start() < emit_from:
            continue
        start_num, start_type, end_num, end_type = int(m.group(1)), m.group(2), int(m.group(3)), m.group(4)
        if start_type != end_type:
            continue
        base = spans.get(m.start(), {})
        for n in range(start_num, end_num + 1):
            if start_type == "호" and base.get("paragraph_type") == "항":
                para, ptype, sub = base["paragraph_num"], "항", n
            else:
                para, ptype, sub = n, start_type, None
            results.append({
                "article_num": base.get("article_num"), "paragraph_num": para,
                "paragraph_type": ptype, "sub_ho": sub,
                "law_name": base.get("law_name"), "doc_name": base.get("doc_name"),
            })
    return results

CITE_LINEBREAK_FIX = re.compile(r"(제\s*\d+)\s*\n\s*([조항호])")
CITE_SPACE_FIX = re.compile(r"제\s+(\d)")      # '「전자서명법」 제 2조제2호', '제1항 제 4호' → 붙여 쓴다

def normalize_citation_linebreaks(text: str) -> str:
    """citation 번호와 단위(조/항/호) 사이에 끼어든 줄바꿈과 '제'·숫자 사이 공백을 제거한다.
    예: '「전자서명법」 제2\n   조제2호' → '「전자서명법」 제2조제2호', '제 2조' → '제2조'
    PDF가 단어 중간에서 줄을 끊는 특성 때문에 생기며, 보정하지 않으면 citation 매치가 통째로 실패한다.
    (강아지 전체 문서 기준 줄바꿈 18건. '제 N' 공백은 법명이 떨어져 자기 약관 조로 잘못 연결되던 원인)
    주의: find_citations_detailed 내부에서만 쓰이며, 저장되는 청크 content는 원문 그대로 유지된다."""
    prev = None
    while prev != text:
        prev = text
        text = CITE_LINEBREAK_FIX.sub(r"\1\2", text)
    return CITE_SPACE_FIX.sub(r"제\1", text)

# ----- 다른 약관 문서를 명시한 인용 (doc_name) -----
# 원문 표기 두 가지: 「반려동물 비용손해 관련 특별약관 일반조항」제15조 (75건) / 보통약관 제35조, …과실로 반려동물 비용손해
# 관련 특별약관 일반조항 제7조 (약 32건). 「」만 보면 괄호 없는 쪽을 놓친다. 공백·줄바꿈은 무시하고 비교한다.
GENERAL_CLAUSE_DOCS = sorted({e["section_name"] for entries in SECTION_MAP.values() for e in entries
                              if e.get("section_name") and e["section_name"].endswith("일반조항")}, key=len, reverse=True)
BOTONG_DOC = re.compile(r"보통약관(?:제\d+절일반조항)?(?:의)?$")       # '보통약관의 제19조' 포함

def extract_doc_name(preceding: str) -> str | None:
    """citation 바로 앞 텍스트가 약관 문서명으로 끝나면 그 문서(section_name, 보통약관은 '보통약관')를 반환."""
    s = re.sub(r"\s+", "", preceding)
    s = re.sub(r"[」』]$", "", s)
    if BOTONG_DOC.search(s):
        return "보통약관"
    s = re.sub(r"의$", "", s)
    for name in GENERAL_CLAUSE_DOCS:
        if s.endswith(re.sub(r"\s+", "", name)):
            return name
    return None

# 열거 연결: 앞 citation과 이 연결자로만 이어지면 같은 출처(법명·문서명)이고, 단독 호는 앞의 항을 잇는다
# 예: 「신용정보법」 제32조‧제33조 / 보통약관 제9조(…), 제10조(…) 및 제38조 / 제1항 제1호 또는 제2호
ENUM_GAP = re.compile(r"^\s*(?:\((?:[^()]|\([^()]*\))*\))?\s*(?:,|·|‧|ㆍ|및|또는|내지|부터|와|과|이나|그리고)\s*$")

# 약관 본문 안의 법령 정의 박스 【법명 제N조(…)】 — 박스 본문의 법명 없는 인용은 그 법의 조문이다
# (예: 【수의사법 제2조(정의)】 안의 '제17조' → 수의사법 제17조. 자기 약관 제17조가 아니다)
LAW_BOX_HEADER = re.compile(r"【\s*[「『｢]?\s*([가-힣·ㆍ\s]+?(?:법률|법|령|규칙))\s*[」』｣]?\s*제(\d+)조(?:의\d+)?\s*\([^)]*\)\s*】")

def law_box_spans(text: str) -> list[tuple[int, int, str, int]]:
    spans = []
    for m in LAW_BOX_HEADER.finditer(text):
        nxt = text.find("【", m.end())
        spans.append((m.end(), nxt if nxt != -1 else len(text), clean_law_name(m.group(1)), int(m.group(2))))
    return spans

def find_citations_detailed(text: str, emit_from: int = 0) -> list[dict]:
    """
    citation을 dict 리스트로 반환.
    - emit_from: 이 위치(줄바꿈 보정 후 기준) 이전의 citation은 내보내지 않는다. 분할된 뒷조각은
      원본 처음부터 조각 끝까지를 넘겨 법명·조번호 맥락(base_law, last_article_num)을 이어받고,
      자기 구간의 citation만 받는다 (cross-part 맥락 단절 해결, 2026-10-12).
    - 조+항/호, 조+호, 단독조, 단독항/호, 단독호 모두 포착
    - 단독 항/호는, 법명이 없는 경우에 한해 직전 조 번호를 이어받는다
      (법명이 있으면 외부 법률이므로 우리 약관의 조 번호를 이어주지 않는다)
    - "부터~까지"/"내지" 범위 표현은 전부 펼친다
    - 동일한 citation은 중복 제거한다
    - 번호와 단위(조/항/호) 사이에 낀 줄바꿈, '제'와 숫자 사이 공백은 추출 직전에 보정한다
    - 약칭 '법/영/규칙 제N조'는 직전 기준법(base_law)을 범주별로 이어받는다
      (법→기준법, 영→기준법+' 시행령', 규칙→기준법+' 시행규칙').
      base_law는 시행령/시행규칙이 아닌 '순수 기본법'일 때만 갱신하고,
      이 약칭 이어받기는 조 번호 전파(last_article_num)에는 영향을 주지 않는다.
    (2026-10-12 추가)
    - doc_name: '보통약관 제N조', '「…일반조항」 제N조'처럼 다른 약관 문서를 명시하면 그 문서명. 03이 그 문서에서만 찾는다.
    - 열거(',', '‧', '및', '또는', '내지'…)로 이어진 인용은 앞 인용의 법명·문서명을 잇고, 단독 호는 앞의 (조, 항)을 잇는다
      → '제1항 제1호 또는 제2호'의 2호는 paragraph_type='항', paragraph_num=1, sub_ho=2 (호 번호를 항으로 두지 않는다).
    - 법령 박스 【법명 제N조(…)】 본문 안의 출처 없는 인용은 그 법의 조문이다(단독 항/호는 박스의 조).
    """
    text = normalize_citation_linebreaks(text)
    boxes = law_box_spans(text)
    def box_at(pos):
        return next(((law, art) for s, e, law, art in boxes if s <= pos < e), None)

    matches = list(CITE_PATTERN_FULL.finditer(text))
    results = []
    spans = {}                       # 매치 시작 위치 → 해석된 citation (범위 펼침이 첫 항목의 맥락을 잇는 데 사용)
    last_article_num = None
    last_law_name = None
    base_law = None
    last_article_law = None          # 직전 조에 딸린 법명(단독 항/호가 상속)
    last_article_doc = None          # 직전 조에 딸린 문서명(단독 항/호가 상속)
    prev = None                      # (직전 매치 끝 위치, 직전 citation)

    for m in matches:
        g1, g2, g3, g4, g5, g6, g7, g8, g9 = m.groups()
        preceding = text[max(0, m.start() - 30):m.start()]
        # 법명 탐색은 60자 — 「환경친화적자동차의 개발 및 보급촉진에 관한 법률」처럼 30자를 넘는 법명이 있다.
        # 약칭('법/영/규칙') 판별은 기존대로 30자(preceding)
        raw_law_name = extract_law_name(text[max(0, m.start() - 60):m.start()])
        law_name = clean_law_name(raw_law_name) if raw_law_name else None
        doc_name = extract_doc_name(text[max(0, m.start() - 60):m.start()]) if law_name is None else None
        box = box_at(m.start())
        enum = prev is not None and bool(ENUM_GAP.match(text[prev[0]:m.start()]))
        # '제43조의2 제1항' — 조의N 바로 뒤의 항/호는 같은 조문의 일부(CITE 패턴이 '의N'을 못 넘어 둘로 끊긴다)
        # '제N조(제목)의 제M항'처럼 조 제목 뒤 '의'로 바로 이어지는 항도 같은 조문의 일부('「…일반조항」제8조(손해방지의무)의 제1항')
        cont = prev is not None and prev[1]["article_num"] is not None and bool(
            re.match(r"^(?:의\s*\d+|\((?:[^()]|\([^()]*\))*\)\s*의)\s*$", text[prev[0]:m.start()]))

        # 약칭 '법/영/규칙' → 직전 기준법 범주별 이어받기 (추출 실패 시에만). 박스 안이면 박스 법의 기본법
        from_abbr = False
        box_base = re.sub(r"\s*시행(령|규칙)$", "", box[0]) if box else None
        if law_name is None and doc_name is None and (base_law or box_base):
            abbr = re.search(r"(?<![가-힣])(법|영|규칙)\s*$", preceding)
            if abbr:
                k = abbr.group(1)
                bl = box_base or base_law     # 박스 안 '법'은 박스 법(상법 시행령 → 상법)이 우선
                law_name = bl if k == "법" else bl + (" 시행령" if k == "영" else " 시행규칙")
                from_abbr = True

        # "동법/같은 법"은 법 이름이 아니라 직전에 언급된 법을 가리키는 대명사이므로 이어받는다
        # ('동법 시행령'은 직전 기본법의 시행령: '보험업법 … 및 동 법 시행령 제43조의2' → 보험업법 시행령)
        pron = re.match(r"^(?:동\s*법|같은\s*법)(\s*시행(?:령|규칙))?$", law_name or "")
        if pron and pron.group(1):
            law_name = (base_law or last_law_name) + " " + pron.group(1).strip() if (base_law or last_law_name) else None
        elif law_name in ("동법", "같은법", "같은 법"):
            law_name = last_law_name
        elif law_name and not from_abbr:
            last_law_name = law_name
            # base_law는 순수 기본법(…법/…법률)일 때만 갱신 — 시행령/시행규칙으론 갱신하지 않는다
            if not re.search(r"시행(령|규칙)$", law_name):
                base_law = law_name

        # 열거로 이어진 인용: 자기 출처가 없으면 앞 인용의 법명·문서명을 잇는다
        if enum and law_name is None and doc_name is None:
            law_name, doc_name = prev[1]["law_name"], prev[1]["doc_name"]
            from_abbr = from_abbr or prev[1].get("_from_abbr", False)
        # 법령 박스 안의 명시 조(제N조…): 그래도 출처가 없으면 박스의 법. 단독 항/호는 아래에서 상속 실패 시에만
        if law_name is None and doc_name is None and box and (g1 or g4 or g6):
            law_name = box[0]

        # 조 번호 전파는 약칭 이어받기의 영향을 받지 않도록 원본 흐름을 유지한다
        has_law_name = (law_name is not None) and not from_abbr

        # 다른 약관 문서를 명시한 조(doc_name)는 뒤의 단독 항에 조 번호를 물려주지 않는다 — 열거로 바로 이어질 때만 잇는다(아래 enum).
        # 예: '「…일반조항」제15조(재가입) 제6항에 따라 연장된 경우에는 … 제6항을 적용하지 않습니다'의 뒤 '제6항'은
        # 자기 조(제1조)의 6항(만료 후 180일)이다. 물려주면 일반조항 제15조 6항으로 잘못 연결된다(2026-10-12).
        inherit = not has_law_name and doc_name is None
        if g1:
            article_num, para_num, para_type, sub_ho = int(g1), int(g2), "항", int(g3) if g3 else None
            if inherit:
                last_article_num, last_article_law, last_article_doc = article_num, law_name, doc_name
        elif g4:
            article_num, para_num, para_type, sub_ho = int(g4), int(g5), "호", None
            if inherit:
                last_article_num, last_article_law, last_article_doc = article_num, law_name, doc_name
        elif g6:
            article_num, para_num, para_type, sub_ho = int(g6), None, None, None
            if inherit:
                last_article_num, last_article_law, last_article_doc = article_num, law_name, doc_name
        elif g7 and cont:
            p = prev[1]
            article_num, para_num, para_type, sub_ho = p["article_num"], int(g7), "항", int(g8) if g8 else None
            law_name, doc_name = p["law_name"], p["doc_name"]
        elif g7:
            article_num = None if has_law_name else last_article_num
            para_num, para_type, sub_ho = int(g7), "항", int(g8) if g8 else None
            if not has_law_name and law_name is None:
                law_name = last_article_law   # 조에 딸린 외부법명 상속 (예: 법 제59조의4 → 제1항)
                doc_name = doc_name or last_article_doc
            if enum and article_num is None and prev[1]["article_num"] is not None:
                article_num = prev[1]["article_num"]     # '제1항부터 제3항' — 앞 인용의 조
            if law_name is None and doc_name is None and box:
                # 박스 안 단독 항인데 이어받을 외부 조가 없으면 박스 조문의 항 (자기 약관 조로 잇지 않는다)
                law_name, article_num = box
        elif g9:
            p = prev[1] if enum else None
            if p and p["paragraph_type"] == "항" and p["paragraph_num"] is not None:
                # '제1항 제1호 또는 제2호' — 앞에서 밝힌 항 안의 호
                article_num, para_num, para_type, sub_ho = p["article_num"], p["paragraph_num"], "항", int(g9)
                law_name, doc_name = p["law_name"], p["doc_name"]
            elif p and p["paragraph_type"] == "호" and p["article_num"] is not None:
                # '제5조제1호 및 제3호' — 앞에서 밝힌 조의 호
                article_num, para_num, para_type, sub_ho = p["article_num"], int(g9), "호", None
                law_name, doc_name = p["law_name"], p["doc_name"]
            else:
                # 이어받을 항·조가 없는 단독 호는 지금 보고 있는 항의 열거 항목('제1호부터 제10호까지')이다.
                # 멀리 앞에서 언급된 조 번호를 잇지 않는다(03에서 self_item, 그래프 제외)
                article_num, para_num, para_type, sub_ho = None, int(g9), "호", None
                law_name = doc_name = None
                from_abbr = False
        else:
            continue

        cit = {
            "article_num": article_num, "paragraph_num": para_num,
            "paragraph_type": para_type, "sub_ho": sub_ho, "law_name": law_name, "doc_name": doc_name,
        }
        prev = (m.end(), {**cit, "_from_abbr": from_abbr})
        spans[m.start()] = cit
        if m.start() < emit_from:      # 앞 조각 구간 — 맥락 갱신만 하고 내보내지 않는다
            continue
        results.append(cit)

    all_results = results + expand_range_citations(text, emit_from, spans)

    # ===== 중복 제거: 모든 필드가 같으면 하나로 =====
    seen = set()
    deduped = []
    for cit in all_results:
        key = (cit["article_num"], cit["paragraph_num"], cit["paragraph_type"], cit["sub_ho"],
               cit["law_name"], cit["doc_name"])
        if key not in seen:
            seen.add(key)
            deduped.append(cit)

    return deduped


def clean_fake_columns(md_table: str) -> str:
    return re.sub(r"\bCol\d+\b", "", md_table)

def trim_to_real_table(md: str) -> str:
    """표 뒤에 섞여든 반복 블록(조항 설명 등)을 잘라내고, 진짜 표 부분만 남긴다."""
    md = clean_fake_columns(md)
    lines = md.split("\n")

    def is_repeat_row(line):
        if not line.strip().startswith("|"):
            return None
        cells = [c.replace("<br>", "").strip() for c in line.strip("|").split("|")]
        non_empty = [c for c in cells if c]
        if not non_empty:
            return True
        return len(set(non_empty)) <= 1

    repeat_count = 0
    cut_at = len(lines)
    run_start = None
    for i, line in enumerate(lines):
        rep = is_repeat_row(line)
        if rep is None:
            continue
        if rep:
            if run_start is None:
                run_start = i
            repeat_count += 1
            if repeat_count >= 3:
                cut_at = run_start
                break
        else:
            repeat_count = 0
            run_start = None

    return "\n".join(lines[:cut_at])

ARTICLE_HEADER = re.compile(r"(?:^|\n{2,})제(\d+)조\(((?:[^()]|\([^()]*\))*?)\)\s*\n(?!\n)", re.MULTILINE | re.DOTALL)

def find_article_headers(sorted_text: str):
    """sort=True로 뽑은 텍스트에서 진짜 조 헤더 목록을 (조번호, 제목, 매치위치) 형태로 반환."""
    return [(int(m.group(1)), m.group(2), m.start()) for m in ARTICLE_HEADER.finditer(sorted_text)]

PARAGRAPH_MARKER = "\uf000"

def split_into_paragraphs(article_text: str):
    """한 조의 본문에서 \uf000 등장 순서대로 1항, 2항...으로 나눈다."""
    parts = article_text.split(PARAGRAPH_MARKER)
    if len(parts) == 1:
        return [{"num": None, "text": article_text.strip()}]
    result = [{"num": None, "text": parts[0].strip()}] if parts[0].strip() else []
    for i, p in enumerate(parts[1:], start=1):
        result.append({"num": i, "text": p.strip()})
    return result

def find_citations(text: str):
    """이 텍스트 안에서 언급된 '제N조', '제N항', '제N호' 인용을 전부 찾아 반환한다."""
    return CITE_PATTERN.findall(text)
    
ARTICLE_HEADER_PREFIX = re.compile(r"^\s*제\d+조(?:\([^)]{0,40}\))?\s*")

def strip_leading_article_header(text: str) -> str:
    """청크 맨 앞의 조 헤더('제1조(목적)')를 떼어낸다.
    이 헤더는 '이 조가 제N조다'라는 제목이지 다른 조를 가리키는 인용이 아니므로,
    citation 추출 전에 제거해 자기참조가 잡히는 것을 막는다.
    (본문 저장에는 영향 없음 — citation 추출용으로만 사용)"""
    return ARTICLE_HEADER_PREFIX.sub("", text, count=1)

In [6]:
# ===== 6: 섹션 전용 함수 =====
def parse_credit_info_notice(text: str) -> list[dict]:
    """'개인신용정보 제공·이용에 대한 고객 권리 안내문' 전용 — 숫자→가나다 2단계, 가나다 레벨이 청크"""
    return split_numbered_then_korean(text)


def parse_meritz_privacy_policy(text: str) -> list[dict]:
    """'메리츠금융그룹 고객정보 취급방침' 전용.
    로마자(I/II/III) 최상위 구분 아래 구조가 제각각이라 블록마다 다르게 처리한다.
    I 블록은 '1. 2.' 목록이지만 그 안에 다시 호 목록('1. 예탁한 금전의 총액')이 섞여 있어
    숫자로 쪼개면 호가 최상위 항목으로 잘못 떨어진다 — 짧은 블록이라 통째로 둔다."""
    chunks = []
    for r in split_by_roman_alpha_title(text):
        if r["roman"] is None:
            # 도입부 + "■ 금융지주회사법 제48조의2" 표 — 원문자(①②) 단위로 나눔
            chunks.extend(split_by_circle_number(r["text"]))
        elif r["roman"] == "I":
            chunks.append({"text": r["text"]})      # 451자, 통째로
        elif r["roman"] == "II":
            chunks.append({"text": r["text"]})      # 서술형, 통째로
        elif r["roman"] == "III":
            chunks.extend(split_by_circle_number(r["text"]))
    return chunks

PAYMENT_FLOW_CLEAN = "보험금청구 사고접수 ➜ 청구서류 안내 ➜ 청구서류 접수 ➜ 보상여부검토 및 사고조사 ➜ 보험금결정 및 지급 ➜ 지급안내문 발송"
def fix_payment_flow_diagram(text: str) -> str:
    """깨진 플로우차트(화살표 다이어그램)를 정리된 한 줄로 치환한다."""
    pattern = re.compile(r"\[\s*보험금\s*지급절차\s*\].*?(?=\n\n\n)", re.DOTALL)
    return pattern.sub(f"[보험금 지급절차]\n{PAYMENT_FLOW_CLEAN}", text)
    
def parse_payment_procedure_notice(text: str) -> list[dict]:
    """'보험금 지급 절차' 전용 — ■ 레벨 청크, 플로우차트 재구성.
    첫 ■ 앞의 머리말(섹션 제목 줄)은 본문이 없어 버린다."""
    text = fix_payment_flow_diagram(text)
    return [c for c in split_by_square_title(text) if len(c["text"].strip()) >= 20]

def parse_misc_notice_p10(text: str) -> list[dict]:
    """10쪽: 안내말씀/유의사항/보험품질보증제/민원상담 — 제목 4개를 직접 지정해서 나눔
    (기호 없는 짧은 안내문이라 범용 패턴으로는 본문과 제목을 구분 못 해, 직접 지정 방식 사용)"""
    titles = ["안내말씀", "유의사항", "보험품질보증제 시행", "보험관련 민원상담 및 분쟁조정 안내"]
    positions = [(t, text.find(t)) for t in titles]
    positions = [(t, p) for t, p in positions if p != -1]
    positions.sort(key=lambda x: x[1])

    chunks = []
    for i, (title, pos) in enumerate(positions):
        end = positions[i+1][1] if i+1 < len(positions) else len(text)
        chunks.append({"title": title, "text": text[pos:end]})
    return chunks
    
def parse_guide_contact_info(text: str) -> list[dict]:
    """'약관 이용 방법' 중 '기타문의사항'만 — 연락처 정보, 통째로 청크 하나
    (1~5번: 약관 읽는 법/QR코드/핵심체크항목 색인표/꿀팁은 메타 안내성이라 스킵.
     체크항목 색인표는 검색이 본문에서 직접 찾아낼 가능성이 높아 가치가 낮다고 판단,
     추후 04번 검색 테스트에서 실제로 누락되는 경로가 보이면 재검토)"""
    idx = text.find("기타문의")
    if idx == -1:
        return []
    return [{"text": text[idx:]}]

# 다수계약 비례보상 수식 — 분수가 세로로 배치돼 추출 시 열이 뒤엉킨다.
# 섹션마다 줄바꿈 위치와 표현("이 계약"/"이 특별약관")이 달라 정규식으로 덩어리째 잡는다.
PROPORTIONAL_FORMULA_MEDICAL = "지급보험금 = 피보험자가 부담한 총 비용금액 × (이 계약의 지급보험금 ÷ 다른 계약이 없는 것으로 하여 각각 계산한 지급보험금의 합계액)"
PROPORTIONAL_FORMULA_LIABILITY = "보상액 = 손해액 × (이 계약의 보상책임액 ÷ 다른 계약이 없는 것으로 하여 각각 계산한 보상책임액의 합계액)"

PROPORTIONAL_MEDICAL_RE = re.compile(
    r"피보험자가[\s\S]{0,40}?부담한[\s\S]{0,40}?×[\s\S]{0,80}?지급보험금의\s*\n?\s*합계액"
)
PROPORTIONAL_LIABILITY_RE = re.compile(
    r"(?:이\s*(?:계약|특별약관)의\s*보상책임액\s*\n\s*)?손해액\s*×[\s\S]{0,80}?보상책임액의\s*\n?\s*합계액"
)


def fix_proportional_formulas(text: str) -> str:
    """다수계약 비례보상 수식(의료비/배상책임)이 세로 분수 배치 때문에 열이 뒤엉켜 추출된 것을
    정리된 한 줄 수식으로 바꾼다. 가입자 유의사항과 특별약관 본문 여러 곳에 같은 수식이 나온다."""
    text = PROPORTIONAL_MEDICAL_RE.sub(PROPORTIONAL_FORMULA_MEDICAL, text)
    text = PROPORTIONAL_LIABILITY_RE.sub(PROPORTIONAL_FORMULA_LIABILITY, text)
    return text


def parse_subscriber_notice(text: str) -> list[dict]:
    """'가입자 유의사항' 전용 — □(대분류) → 1.2.(중분류, 있을 때만) → ○(소분류) 3단계.
    다수계약 비례보상 수식은 정리된 한 줄로 치환하고,
    맨 끝의 정형 문구("이 가입자 유의사항은 ... 약관의 내용을 따릅니다")는 버린다."""
    text = fix_proportional_formulas(text)
    tail = text.find("이 가입자 유의사항은")
    if tail != -1:
        text = text[:tail]

    chunks = []
    for sq in split_by_square_box(text):
        if sq["title"] is None:
            continue
        for num_seg in split_by_numbered_title(sq["text"]):
            for c in split_by_circle_bullet(num_seg["text"]):
                if len(c["text"].strip()) < 30:
                    continue
                chunks.append({
                    "square_title": sq["title"],
                    "item_num": num_seg["num"],
                    "item_title": num_seg["title"],
                    "circle_title": c["title"],
                    "text": c["text"],
                })
    return chunks
    
def fix_split_page_number_glitches(text: str) -> str:
    """'주요내용 요약서'에서 페이지 번호가 항목번호 앞에 줄바꿈 없이 붙어
    숫자목록 매치를 놓치는 확인된 2건을 직접 보정한다.
    (강아지 42쪽→13번, 44쪽→17번. 고양이는 페이지가 달라 별도 확인 필요)"""
    text = text.replace("4213. 계약 전", "13. 계약 전")
    text = text.replace("4417. 대위권", "17. 대위권")
    return text


def parse_summary_notice(text: str) -> list[dict]:
    """'주요내용 요약서' 전용 — 숫자목록(1~17) 레벨에서만 청크 분할.
    하위 구조(①②, 1), (1))는 쪼개지 않고 본문에 포함한다.
    첫 항목 앞의 머리말(섹션 제목 줄)은 본문이 없어 버린다."""
    text = fix_split_page_number_glitches(text)
    return [c for c in split_by_numbered_title(text) if len(c["text"].strip()) >= 20]

def parse_terms_glossary(text: str) -> list[dict]:
    """'보험용어 해설' 전용 — ○ 레벨이 곧 용어 하나(청크 하나).
    용어 설명 안의 1)2)·①② 하위 구조는 쪼개지 않고 본문에 포함한다."""
    chunks = []
    for seg in split_by_circle_bullet(text):
        if len(seg["text"].strip()) < 20:
            continue
        chunks.append(seg)
    return chunks

def parse_articles(pages: list[dict]) -> list[dict]:
    """조-항 구조 섹션(보통약관/특별약관) 전용 파서.
    해당 구간의 페이지 리스트를 받아 조 헤더로 자르고, 유형(고급형/기본형/실속형)과
    항(\uf000)으로 분리한 청크를 반환한다.
    각 청크의 페이지는 글자 위치를 실제 페이지 경계와 대조해 구한다(비율 추정 아님).
    표를 위치 기준으로 붙일 수 있도록 start_offset/end_offset도 함께 내보낸다."""

    full_text = ""
    boundaries = []
    for p in pages:
        boundaries.append((len(full_text), p["page"]))
        full_text += p["text"] + "\n"

    def offset_to_page(offset: int) -> int:
        result = boundaries[0][1]
        for start_offset, page_num in boundaries:
            if start_offset <= offset:
                result = page_num
            else:
                break
        return result

    headers = find_article_headers(full_text)
    articles = []
    for idx, (num, title, pos) in enumerate(headers):
        end = headers[idx + 1][2] if idx + 1 < len(headers) else len(full_text)
        attach = ATTACHMENT_MARKER.search(full_text, pos, end)
        if attach:
            end = min(end, attach.start())
        articles.append({"article_num": num, "article_title": title,
                         "start_offset": pos, "end_offset": end})
    
    chunks = []
    for a in articles:
        art_text = full_text[a["start_offset"]:a["end_offset"]]
        seg_cursor = a["start_offset"]

        for seg in split_by_coverage_type(art_text):
            seg_start = seg_cursor
            seg_cursor += len(seg["text"])

            parts = seg["text"].split(PARAGRAPH_MARKER)
            part_cursor = seg_start

            for i, part in enumerate(parts):
                part_start = part_cursor
                part_cursor += len(part) + (len(PARAGRAPH_MARKER) if i < len(parts) - 1 else 0)
                stripped = strip_leading_article_header(part).strip()
                if not stripped or COVERAGE_LABEL_ONLY.match(stripped):
                    continue
                chunks.append({
                    "article_num": a["article_num"],
                    "article_title": a["article_title"],
                    "coverage_type": seg["coverage_type"],
                    "paragraph_num": i if i > 0 else None,
                    "text": part.strip(),
                    "start_offset": part_start,
                    "end_offset": part_cursor,
                    "start_page": offset_to_page(part_start),
                    "end_page": offset_to_page(max(part_start, part_cursor - 1)),
                })
    for c in chunks:
        c["text"] = fix_proportional_formulas(c["text"])
    return chunks
    
def parse_medical_teukyak_2stage(pages: list[dict]) -> list[dict]:
    """통원/입원의료비 Ⅰ·Ⅱ 보장 특약 전용 — 조-항 파싱 후 깨진 한도표를 재구성한다.
    pymupdf가 이 한도표를 제대로 추출하지 못해(세로쓰기 라벨 '통원/의료비'가 셀 사이에 끼어듦)
    본문 텍스트에서 숫자를 정규식으로 뽑아 읽을 수 있는 표로 다시 만든다.
    2단계 = 수술한 날 / 수술하지 않은 날."""
    chunks = parse_articles(pages)
    for c in chunks:
        if c["paragraph_num"] == 2:
            c["text"] = rebuild_level2_table(c["text"])
    return chunks


def parse_medical_teukyak_3stage(pages: list[dict]) -> list[dict]:
    """통원/입원의료비 Ⅲ 보장 특약 전용 — 조-항 파싱 후 깨진 한도표를 재구성한다.
    Ⅲ유형은 MRI/CT·내시경처치 여부와 연간 횟수까지 조건이 갈려 4단계 구조라 별도 함수를 쓴다."""
    chunks = parse_articles(pages)
    for c in chunks:
        if c["paragraph_num"] == 2:
            c["text"] = rebuild_level3_table(c["text"])
    return chunks

def parse_teukjeong_table(pages: list[dict], pdf_path) -> list[dict]:
    """특정질병 분류표(별첨) 전용 — 조-항 구조가 아니라 표 자체가 내용이다.
    find_tables()로 표를 직접 읽어, 분류번호/질병분류명이 비어 있는 행은 직전 값을 이어받아 채운 뒤
    (분류번호, 질병분류명) 단위로 묶어 '질병코드: 질병명' 목록 한 덩어리를 만든다."""
    pdf = pymupdf.open(pdf_path)
    all_rows = []
    for p in pages:
        for t in pdf[p["page"]].find_tables().tables:
            rows = t.extract()
            all_rows.extend(rows[1:])          # 헤더 행 제외
    pdf.close()

    current_num, current_name = None, None
    filled = []
    for row in all_rows:
        num, name, code, item = row
        num = (num or "").strip() if num else None
        name = (name or "").replace("\n", "") if name else None
        if num:
            current_num, current_name = num, name
        filled.append({"num": current_num, "category": current_name, "code": code, "item": item})

    groups = defaultdict(list)
    for r in filled:
        groups[(r["num"], r["category"])].append(r)

    start_page, end_page = pages[0]["page"], pages[-1]["page"]
    chunks = []
    for (num, category), items in groups.items():
        lines = [f"- {it['code']}: {it['item'].replace(chr(10), ' ')}" for it in items]
        chunks.append({
            "article_num": None, "article_title": None,
            "coverage_type": None, "paragraph_num": None,
            "item_num": num, "item_title": category,
            "text": f"{category} (관련 질병코드 {len(items)}개)\n" + "\n".join(lines),
            "start_page": start_page, "end_page": end_page,
        })
    return chunks

def parse_table_only_section(pages: list[dict]) -> list[dict]:
    """표 하나가 내용 전부인 짧은 섹션 전용 (별표1·3·4·5).
    본문 텍스트는 열이 뒤엉켜 읽기 어려우므로 맥락용으로만 두고,
    실제 내용은 extract_tables_with_offset이 붙이는 표 markdown이 담당한다."""
    text = "\n".join(p["text"] for p in pages)
    return [{
        "article_num": None, "article_title": None,
        "coverage_type": None, "paragraph_num": None,
        "text": text.strip(),
        "start_offset": 0, "end_offset": len(text) + 10,
        "start_page": pages[0]["page"], "end_page": pages[-1]["page"],
    }]

def parse_byeopyo2(pages: list[dict]) -> list[dict]:
    """【별표2】 장해분류표 전용 — \uf000(총칙/판정기준) → 숫자(신체부위) → 가나다(분류/판정기준) 3단계.
    가나다 레벨이 청크이며, 하위 항목이 없는 신체부위는 숫자 레벨이 통째로 청크가 된다.
    표(장해의 분류/지급률)를 위치로 붙일 수 있도록 start_offset/end_offset을 함께 낸다."""
    full_text = ""
    boundaries = []
    for p in pages:
        boundaries.append((len(full_text), p["page"]))
        full_text += p["text"] + "\n"

    def offset_to_page(offset: int) -> int:
        result = boundaries[0][1]
        for start_offset, page_num in boundaries:
            if start_offset <= offset:
                result = page_num
            else:
                break
        return result

    chunks = []
    for top in split_into_paragraphs(full_text):
        top_title = "총칙" if top["num"] == 1 else ("장해분류별 판정기준" if top["num"] == 2 else None)
        if top_title is None:
            continue
        for mid in split_by_numbered_title(top["text"]):
            if mid["num"] is None:
                continue
            subs = split_by_korean_sub(mid["text"])
            targets = [(None, None, mid["text"])] if (len(subs) == 1 and subs[0]["label"] is None) \
                      else [(s["label"], s["title"], s["text"]) for s in subs if s["label"]]
            for label, sub_title, text in targets:
                start = full_text.find(text)
                end = start + len(text) if start != -1 else len(full_text)
                chunks.append({
                    "article_num": None, "article_title": None,
                    "coverage_type": None, "paragraph_num": None,
                    "top_category": top_title,
                    "item_num": mid["num"], "item_title": mid["title"],
                    "sub_label": label, "sub_title": sub_title,
                    "text": text.strip(),
                    "start_offset": start, "end_offset": end,
                    "start_page": offset_to_page(start) if start != -1 else pages[0]["page"],
                    "end_page": offset_to_page(end - 1) if start != -1 else pages[-1]["page"],
                })
    return chunks


def parse_bupgyu(pages: list[dict]) -> list[dict]:
    """법규 섹션(인용 법률 전문) 전용.
    조 헤더는 약관과 같은 '제N조(제목)' 형식이지만, 항 마커가 약관의 \uf000이 아니라
    실제 원문자(①②③)다 — 기존 파이프라인은 이를 약관과 똑같이 취급해 항 분리를 못 했다.
    발췌본이라 항이 ①부터 시작하지 않는 조가 있으므로(금융소비자보호법 제17조는 ②부터),
    등장 순서로 세지 않고 원문자 자체를 항 번호로 읽는다.
    항이 없는 조(민법 제753조 등)는 조 전체가 청크 하나가 된다."""
    full_text = ""
    boundaries = []
    for p in pages:
        boundaries.append((len(full_text), p["page"]))
        full_text += p["text"] + "\n"

    def offset_to_page(offset: int) -> int:
        result = boundaries[0][1]
        for start_offset, page_num in boundaries:
            if start_offset <= offset:
                result = page_num
            else:
                break
        return result

    headers = find_article_headers(full_text)
    
    chunks = []
    for idx, (num, title, pos) in enumerate(headers):
        end = headers[idx + 1][2] if idx + 1 < len(headers) else len(full_text)
        art_text = full_text[pos:end]

        cursor = pos
        for part in PARAGRAPH_CIRCLE_SPLIT.split(art_text):
            part_start = cursor
            cursor += len(part)
            stripped = strip_leading_article_header(part).strip()
            if not stripped or EMPTY_PARAGRAPH.match(stripped):     
                continue
            first = part.strip()[:1]
            chunks.append({
                "article_num": num, "article_title": title,
                "coverage_type": None,
                "paragraph_num": CIRCLE_TO_NUM.get(first),
                "text": part.strip(),
                "start_offset": part_start, "end_offset": cursor,
                "start_page": offset_to_page(part_start),
                "end_page": offset_to_page(max(part_start, cursor - 1)),
            })
    return chunks

In [7]:
# ===== 7: 페이지 로드 + 섹션별 파서 호출 =====

def strip_page_number(text: str, page_index: int) -> str:
    """쪽 하단의 쪽 번호 줄을 지운다. 쪽 번호는 각 쪽의 마지막 비어 있지 않은 줄이고 값이 PDF 쪽 index와 같다
    (2026-10-12 측정: 강아지 292쪽 중 290, 고양이 266쪽 중 264 — 나머지는 쪽 번호 없는 표지).
    지우지 않으면 쪽을 넘어가는 문장 한가운데 숫자가 끼어('청약서상 기재사항이\n 69\n사실과 다른') 검색·FAQ·답변에 섞인다.
    값이 쪽 번호와 같을 때만 지우므로 본문의 숫자(지급률 100, 60 등)는 건드리지 않는다."""
    lines = text.split("\n")
    for k in range(len(lines) - 1, -1, -1):
        if lines[k].strip():
            if lines[k].strip() == str(page_index):
                lines[k] = ""
            break
    return "\n".join(lines)


def load_pages(species: str) -> list[dict]:
    """페이지별 원문 텍스트를 읽어온다 (sort=True로 읽기 순서 보정, 쪽 번호 줄 제거)."""
    doc = pymupdf.open(PDF_FILES[species])
    pages = [{"page": i, "text": strip_page_number(doc[i].get_text(sort=True), i)} for i in range(len(doc))]
    doc.close()
    return pages


pages_dog = load_pages("강아지")
pages_cat = load_pages("고양이")


# 파서는 받는 인자에 따라 세 종류다
PAGE_BASED_PARSERS = {          # pages를 받음 — 글자 위치→페이지 추적이 필요
    "parse_articles", "parse_medical_teukyak_2stage", "parse_medical_teukyak_3stage",
    "parse_bupgyu", "parse_byeopyo2", "parse_table_only_section",
}
PDF_BASED_PARSERS = {           # pages + pdf_path를 받음 — find_tables()로 표를 직접 읽음
    "parse_teukjeong_table",
}
# 재구성 표를 본문에 직접 넣는 파서 — pymupdf가 뜬 원본 표를 덧붙이면 중복되므로 표 첨부를 생략한다
REBUILD_TABLE_PARSERS = {"parse_medical_teukyak_2stage", "parse_medical_teukyak_3stage"}
# 나머지(안내류)는 text 하나만 받음


def extract_doc_type(section_name: str) -> str | None:
    """section_name에서 Ⅰ/Ⅱ/Ⅲ 유형을 추출. 의료비보장 계열이 아니면 구분 자체가 없으므로 None."""
    if not section_name or "의료비" not in section_name:
        return None
    if "Ⅲ" in section_name:
        return "Ⅲ"
    elif "Ⅱ" in section_name:
        return "Ⅱ"
    return "Ⅰ"


def build_chunks(species: str) -> list[dict]:
    """SECTION_MAP을 순회하며 각 구간의 전용 파서를 호출해 청크를 만든다.
    표는 pages 기반 파서에만 붙인다(start_offset/end_offset이 있어야 위치로 매칭 가능).
    section/section_name/type은 호출하는 쪽이 SECTION_MAP에서 바로 붙인다.
    article_block_id는 전체를 합친 뒤 '섹션+조' 단위로 매긴다 — 같은 조의 여러 항이 같은 id를
    공유해야 03번이 단독 항 참조('제1항')를 같은 조 안에서 해석할 수 있다."""
    src = pages_dog if species == "강아지" else pages_cat
    pdf_path = PDF_FILES[species]
    all_chunks = []

    for entry in SECTION_MAP[species]:
        name = entry.get("parser")
        if not name:
            continue
        fn = globals()[name]
        pages = [p for p in src if entry["start"] <= p["page"] <= entry["end"]]
        if not pages:
            continue

        if name in PDF_BASED_PARSERS:
            chunks = fn(pages, pdf_path)
        elif name in PAGE_BASED_PARSERS:
            chunks = fn(pages)
            # 의료비 특약(Ⅰ·Ⅱ·Ⅲ)은 rebuild_levelN_table이 재구성한 한도표를 이미 본문에 넣으므로,
            # pymupdf가 뜬 원본 한도표를 다시 붙이면 같은 숫자가 두 번 들어간다 → 표 첨부 생략
            if name not in REBUILD_TABLE_PARSERS:
                tables = extract_tables_with_offset(pages, pdf_path, verbose=False)
                chunks = attach_tables_by_offset(chunks, tables)
        else:
            chunks = fn("\n".join(p["text"] for p in pages))

        for c in chunks:
            c.setdefault("content", c.get("text", ""))
            c.setdefault("start_page", entry["start"])
            c.setdefault("end_page", entry["end"])
            c["section"] = entry["section"]
            c["section_name"] = entry["section_name"]
            c["type"] = extract_doc_type(entry["section_name"])
        all_chunks.extend(chunks)

    block_id = -1
    prev_key = None
    for c in all_chunks:
        key = (c["section_name"], c.get("article_num"))
        if key != prev_key:
            block_id += 1
            prev_key = key
        c["article_block_id"] = block_id

    return all_chunks


chunks_dog = build_chunks("강아지")
chunks_cat = build_chunks("고양이")
print("강아지 청크:", len(chunks_dog))
print("고양이 청크:", len(chunks_cat))

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


강아지 청크: 890
고양이 청크: 812


In [8]:
# ===== 8: 최종 레코드 조립 =====

# ----- 분할 정책 (2026-10-12 변경: 1000자 글자 단위 + 200자 오버랩 → 구조 경계 분할, 오버랩 없음) -----
# 글자 단위 분할은 절단점 44/45가 문장 중간, 17/18이 단어 중간이었고 법령 박스(118조의4)를 가로질렀다.
# 오버랩은 fetch 결합 시 같은 문장이 두 번 들어가게 했다. → 목표 1000자 근처의 구조 경계에서 자르고 겹치지 않는다.
MAX_CHUNK_LEN = 1000            # 목표 길이 — 이 값에 가장 가까운 경계를 고른다
SPLIT_WINDOW = (600, 1200)      # 경계를 찾는 범위. 남은 길이가 상한 이하면 더 자르지 않는다
TABLE_EXEMPT_LEN = 3000         # [표]가 붙은 청크는 이 길이까지 통째로 둔다(표를 쪼개지 않기 위해)

# 우선순위 순 — 앞 단계에서 창 안에 후보가 하나라도 있으면 뒤 단계는 보지 않는다
SPLIT_BOUNDARIES = [
    ("법령박스", re.compile(r"\n(?=[ \t]*【)")),
    ("항",       re.compile(r"\n(?=[ \t]*[①-⑳])")),
    ("호",       re.compile(r"\n(?=[ \t]*(?:\d{1,2}\.|\(\d{1,2}\)|\d{1,2}\))[ \t])")),
    ("목",       re.compile(r"\n(?=[ \t]*[가-하]\.[ \t])")),
    ("문장끝",   re.compile(r"(?<=다\.)\s+|(?<=요\.)\s+")),
    ("빈줄",     re.compile(r"\n[ \t]*\n")),
    ("줄바꿈",   re.compile(r"\n")),
]
AFTER_MATCH = {"문장끝", "빈줄"}   # 매치 끝에서 자름(공백을 앞 조각에). 나머지는 표지 줄 시작에서 자름
# 절단점 바로 뒤가 페이지 번호 줄이면 그 줄까지 앞 조각에 넘긴다 — 뒷조각이 '236\n<붙임>'처럼 숫자로 시작하지 않게
LEADING_PAGE_NUM = re.compile(r"[ \t\n]*\d{1,4}[ \t]*\n")

def split_long_content(content: str, target=MAX_CHUNK_LEN, window=SPLIT_WINDOW):
    """content를 구조 경계에서 나눈다. 반환: (조각 리스트, 각 절단점의 경계 종류 리스트).
    조각을 이어 붙이면 원문과 정확히 같다(오버랩 없음)."""
    lo, hi = window
    if "[표]" in content and len(content) <= TABLE_EXEMPT_LEN:
        return [content], []
    parts, kinds, s = [], [], 0
    while len(content) - s > hi:
        win = content[s + lo:s + hi + 1]
        best = None
        for name, pat in SPLIT_BOUNDARIES:
            cands = [s + lo + (m.end() if name in AFTER_MATCH else m.start() + 1) for m in pat.finditer(win)]
            cands = [x for x in cands if s + lo <= x <= s + hi]
            if cands:
                best = (name, min(cands, key=lambda x: abs(x - (s + target))))
                break
        if best is None:                       # 창 안에 줄바꿈조차 없을 때만 — 측정상 0건
            best = ("강제", s + target)
        cut = best[1]
        page = LEADING_PAGE_NUM.match(content, cut)
        if page:
            cut = page.end()
        parts.append(content[s:cut])
        kinds.append(best[0])
        s = cut
    parts.append(content[s:])
    return parts, kinds


# ----- 뒷조각 맥락 보완 (context_prefix) -----
# 오버랩을 없애면 뒷조각이 '9. …'처럼 열거 항목으로 시작할 때 무엇의 열거인지 사라진다.
# content에 붙이면 fetch 결합 시 중복되므로 별도 필드에 두고, 임베딩/BM25/리랭커 헤더와 LLM 컨텍스트 표시에만 쓴다.
#   ① 같은 항에 '다음 각 호/목' 류 도입 문장이 있으면 그 문장 전체 (예: 신용정보법 제32조 ⑥)
#   ② 01이 채운 제목 메타(item_*/sub_*)가 있으면 제목 경로 (예: 별표2 '1. 눈의 장해 > 나. 장해판정기준')
#   ③ 본문 속 가장 가까운 'N. 제목 > 가. 제목' 경로 (예: 금융소비자보호법 제19조 '… > 라. 대출성 상품')
# 길이 상한은 두지 않는다(측정 최대 111자) — 200자는 비정상 입력 방어용 안전장치.
CONTEXT_PREFIX_SAFETY = 200
ENUM_ITEM_START = re.compile(r"^[ \t]*(?:\d{1,2}\.|\(\d{1,2}\)|\d{1,2}\)|[가-하]\.)[ \t]")
HANG_LINE = re.compile(r"^[ \t]*[①-⑳]", re.M)
HO_LINE = re.compile(r"^[ \t]*(?:\d{1,2}\.|\(\d{1,2}\)|\d{1,2}\))[ \t]", re.M)
ENUM_LEAD = re.compile(r"다음 각 [호목]|다음 중|아래의 경우|아래와 같")
H1_LINE = re.compile(r"^[ \t]*(\d{1,2})\.[ \t]+(\S[^\n]{0,25})$", re.M)   # 짧은 제목줄 '1. 눈의 장해'
H2_LINE = re.compile(r"^[ \t]*([가-하])\.[ \t]+(\S[^\n]{0,25})$", re.M)
PAGE_NUM_LINE = re.compile(r"^\s*\d{1,4}\s*$")

def _flatten(text: str) -> str:
    """페이지 번호 줄을 버리고 줄바꿈·연속 공백을 한 칸으로 합친다."""
    lines = [l for l in text.splitlines() if not PAGE_NUM_LINE.match(l)]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

def _safety_cap(text: str) -> str:
    return text if len(text) <= CONTEXT_PREFIX_SAFETY else "…" + text[-(CONTEXT_PREFIX_SAFETY - 1):]

def enumeration_lead_in(content: str, pos: int):
    """pos에서 시작하는 열거 항목이 속한 항의 '다음 각 호' 류 도입 문장. 없으면 None."""
    hangs = [m.start() for m in HANG_LINE.finditer(content, 0, pos)]
    if not hangs:
        return None
    first_item = HO_LINE.search(content, hangs[-1], pos)
    block = _flatten(content[hangs[-1]:first_item.start() if first_item else pos])
    sentences = [s for s in re.split(r"(?<=다\.)\s+", block) if ENUM_LEAD.search(s)]
    return sentences[-1] if sentences else None

def metadata_heading_path(c: dict):
    """01 파서가 채운 제목 메타(별표·별첨·안내류)로 만든 경로. 없으면 None."""
    path = []
    if c.get("item_title"):
        path.append(f"{c['item_num']}. {c['item_title']}" if c.get("item_num") is not None else c["item_title"])
    if c.get("sub_title"):
        path.append(f"{c['sub_label']}. {c['sub_title']}" if c.get("sub_label") else c["sub_title"])
    return " > ".join(_flatten(p) for p in path) or None

def text_heading_path(content: str, pos: int):
    """본문에서 pos 앞의 가장 가까운 'N. 제목'과 그 아래 '가. 제목'으로 만든 경로. 없으면 None."""
    h1 = list(H1_LINE.finditer(content, 0, pos))
    if not h1:
        return None
    path = [_flatten(h1[-1].group(0))]
    h2 = list(H2_LINE.finditer(content, h1[-1].end(), pos))
    if h2:
        path.append(_flatten(h2[-1].group(0)))
    return " > ".join(path)

def build_context_prefix(c: dict, content: str, part: str, pos: int):
    """분할된 뒷조각(pos>0) 하나의 context_prefix와 출처 종류. 첫 조각·미분할 청크는 (None, None)."""
    if pos == 0:
        return None, None
    starts_enum = bool(ENUM_ITEM_START.match(part.lstrip("\n")))
    if starts_enum:
        lead = enumeration_lead_in(content, pos)
        if lead:
            return _safety_cap(lead), "도입문장"
    path = metadata_heading_path(c)
    if path:
        return _safety_cap(path), "제목메타"
    if starts_enum:
        path = text_heading_path(content, pos)
        if path:
            return _safety_cap(path), "본문제목"
    return None, None

DOC_META = {
    "강아지": {"doc_id": "petmint_dog_2604", "product_name": "무배당 펫퍼민트 Puppy&Family보험 다이렉트2604"},
    "고양이": {"doc_id": "petmint_cat_2604", "product_name": "무배당 펫퍼민트 Cat&Family보험 다이렉트2604"},
}
POLICY_VERSION = "3.0.0"
EFFECTIVE_DATE = "2026-07-13"
EXPIRY_DATE = None

def build_final_records(chunks: list[dict], species: str) -> list[dict]:
    meta = DOC_META[species]
    records = []
    for c in chunks:
        # 법규 섹션의 law_name 없는 citation은 그 법 자신의 조문을 가리킨다
        # (예: 민사소송법 제254조 안의 "제249조제1항" → 민사소송법 제249조)
        self_law = c["section_name"].split("】")[-1].strip() if c["section"] == "법규" else None

        parts, kinds = split_long_content(c["content"])
        assert "".join(parts) == c["content"]          # 오버랩 없음: 조각 결합 = 원문
        SPLIT_STATS["boundary"].update(kinds)
        total = len(parts)
        pos = 0
        for idx, part in enumerate(parts):
            context_prefix, prefix_kind = build_context_prefix(c, c["content"], part, pos)
            if context_prefix:
                SPLIT_STATS["prefix"][prefix_kind] += 1
            # citation은 원본 처음~이 조각 끝까지 읽어 맥락을 잇고, 이 조각 구간의 것만 받는다
            upto = strip_leading_article_header(c["content"][:pos + len(part)])
            emit_from = len(normalize_citation_linebreaks(strip_leading_article_header(c["content"][:pos]))) if pos else 0
            citation_details = find_citations_detailed(upto, emit_from=emit_from)
            pos += len(part)
            if self_law:
                for cit in citation_details:
                    if cit["law_name"] is None:
                        cit["law_name"] = self_law
            records.append({
                "doc_id": meta["doc_id"],
                "product_name": meta["product_name"],
                "policy_version": POLICY_VERSION,
                "effective_date": EFFECTIVE_DATE,
                "expiry_date": EXPIRY_DATE,
                "species": species,
                "section": c["section"],
                "section_name": c["section_name"],
                "article_block_id": c.get("article_block_id"),
                "article_num": c.get("article_num"),
                "article_title": c.get("article_title"),
                "coverage_type": c.get("coverage_type"),
                "type": c.get("type"),
                "paragraph_num": c.get("paragraph_num"),
                "top_category": c.get("top_category"),
                "item_num": c.get("item_num"),
                "item_title": c.get("item_title"),
                "sub_label": c.get("sub_label"),
                "sub_title": c.get("sub_title"),
                "chunk_index": idx,
                "chunk_total": total,
                "citations": citation_details,
                "printed_page_range": [c["start_page"], c["end_page"]],
                "context_prefix": context_prefix,
                "content": part,
            })
    return records

from collections import Counter

for species, src in [("강아지", chunks_dog), ("고양이", chunks_cat)]:
    SPLIT_STATS = {"boundary": Counter(), "prefix": Counter()}
    recs = build_final_records(src, species)
    if species == "강아지":
        records_dog = recs
    else:
        records_cat = recs
    n_split = sum(1 for r in recs if r["chunk_index"] == 0 and r["chunk_total"] > 1)
    lens = sorted(len(r["content"]) for r in recs if r["chunk_total"] > 1)
    print(f"{species} 최종 레코드 수: {len(recs)} | 분할된 원본 {n_split}개 → 조각 {len(lens)}개")
    print(f"   절단 경계: {dict(SPLIT_STATS['boundary'])}")
    print(f"   조각 길이 min {lens[0] if lens else '-'} / max {lens[-1] if lens else '-'}"
          f" | context_prefix: {dict(SPLIT_STATS['prefix'])}")

강아지 최종 레코드 수: 928 | 분할된 원본 26개 → 조각 64개
   절단 경계: {'항': 8, '법령박스': 10, '호': 14, '문장끝': 3, '빈줄': 3}
   조각 길이 min 207 / max 1083 | context_prefix: {'제목메타': 18, '본문제목': 1, '도입문장': 1}
고양이 최종 레코드 수: 850 | 분할된 원본 26개 → 조각 64개
   절단 경계: {'항': 8, '법령박스': 10, '호': 14, '문장끝': 3, '빈줄': 3}
   조각 길이 min 207 / max 1083 | context_prefix: {'제목메타': 18, '본문제목': 1, '도입문장': 1}


In [9]:
# ===== 9: 최종 레코드 저장 =====

import json
from pathlib import Path

CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
with open(CHUNKS_DIR / "petmint_강아지.json", "w", encoding="utf-8") as f:
    json.dump(records_dog, f, ensure_ascii=False, indent=2)
print("저장 완료:", len(records_dog), "건")

with open(CHUNKS_DIR / "petmint_고양이.json", "w", encoding="utf-8") as f:
    json.dump(records_cat, f, ensure_ascii=False, indent=2)
print("저장 완료:", len(records_cat), "건")

저장 완료: 928 건
저장 완료: 850 건


In [10]:
# ===== 10: 한도 기준표(limit_table.py) 생성 =====
# 통원/입원 Ⅰ·Ⅱ·Ⅲ 지급한도 — 01번 원문(SECTION_MAP 범위) 직접 대조 완료, 강아지/고양이 동일 확인됨
# 보험금 지급금액 산출 공식 — 약관 요약서 원문 예시 5건으로 검증 완료

limit_table_code = '''"""
펫퍼민트 반려견/반려묘 약관 — 통원/입원 Ⅰ·Ⅱ·Ⅲ 지급한도 기준표
01번 전처리 단계에서 원문 대조로 확정 (강아지/고양이 동일 구조 확인됨)
04번(RAG)과 06번(품질 검증)이 공유해서 참조하는 "진실의 원천" 파일
"""

# 자기부담금: 통원/입원 Ⅰ·Ⅱ·Ⅲ 전부 공통, 비수술일/수술일 구분 없이 동일 적용
# 3만원 또는 5만원 중 "보험증권에 기재된" 값 — 가입 시 선택에 따라 결정되므로 단일 확정값 아님
COPAY_OPTIONS = [30_000, 50_000]

LIMIT_TABLE = {
    # (coverage_type, type_): {non_surgery, surgery, rate, annual_total}  -- Ⅰ·Ⅱ 유형용 (통원/입원 공통)
    # non_surgery/surgery 공통 계산식: (실제 치료비 - 자기부담금) × rate, 1일당 한도까지
    # non_surgery: 수술을 하지 않은 날, 1일당 / surgery: 수술을 한 날(당일에 한함), 1일당
    ("기본형", "Ⅰ"): {"non_surgery": 150_000,  "surgery": 2_000_000, "rate": 0.7, "annual_total": 5_000_000},
    ("고급형", "Ⅰ"): {"non_surgery": 300_000,  "surgery": 2_000_000, "rate": 0.7, "annual_total": 7_500_000},
    ("실속형", "Ⅰ"): {"non_surgery": 100_000,  "surgery": 1_500_000, "rate": 0.5, "annual_total": 3_500_000},
    ("기본형", "Ⅱ"): {"non_surgery": 150_000,  "surgery": 2_500_000, "rate": 0.7, "annual_total": 10_000_000},
    ("고급형", "Ⅱ"): {"non_surgery": 300_000,  "surgery": 2_500_000, "rate": 0.7, "annual_total": 15_000_000},
    ("실속형", "Ⅱ"): {"non_surgery": 100_000,  "surgery": 2_000_000, "rate": 0.5, "annual_total": 7_000_000},
}

LIMIT_TABLE_TYPE3 = {
    # coverage_type: {...}  -- Ⅲ 유형용 (통원/입원 공통)
    # 자기부담금(COPAY_OPTIONS)은 아래 모든 scenario에 공통 적용, rate(보상비율)도 공통 적용
    # 상위 조건 1) 수술을 하지 않은 날 — 그 안에서 다시 3가지로 세분화됨:
    #   non_surgery_mri_1st:      수술 안 한 날 중, MRI·CT·내시경처치 받은 날의 "연간 첫 번째"
    #   non_surgery_mri_2nd_plus: 수술 안 한 날 중, MRI·CT·내시경처치 받은 날의 "연간 두 번째 이상"
    #   non_surgery_no_mri:       수술 안 한 날 중, MRI·CT·내시경처치를 받지 않은 날
    # 상위 조건 2) 수술을 한 날 — 수술당일에 한하여 적용되는 단일 한도:
    #   surgery:                  수술을 한 날(당일에 한함), 1일당
    # 중요: MRI/CT/내시경처치와 수술이 "같은 날" 동시에 발생하면,
    #       MRI 한도가 아니라 surgery(수술) 한도가 우선 적용된다 (원문 예시로 확인됨)
    "기본형": {"non_surgery_mri_1st": 500_000,   "non_surgery_mri_2nd_plus": 150_000, "non_surgery_no_mri": 150_000, "surgery": 2_500_000, "rate": 0.7, "annual_total": 10_000_000},
    "고급형": {"non_surgery_mri_1st": 1_000_000, "non_surgery_mri_2nd_plus": 300_000, "non_surgery_no_mri": 300_000, "surgery": 2_500_000, "rate": 0.7, "annual_total": 15_000_000},
    "실속형": {"non_surgery_mri_1st": 300_000,   "non_surgery_mri_2nd_plus": 100_000, "non_surgery_no_mri": 100_000, "surgery": 2_000_000, "rate": 0.5, "annual_total": 7_000_000},
}

# 자기부담금(3만원/5만원 선택)이 공통 적용되지 않는 별도 담보 — 건별로 고정 금액
OTHER_COPAY = {
    "갱신형 펫퍼민트 반려견 배상책임": {"copay": 30_000, "basis": "매 사고당"},
    "민사소송법률비용Ⅱ": {"copay": 100_000, "basis": "자기부담금"},
}


def get_limit(coverage_type: str, type_: str, scenario: str) -> dict:
    """
    coverage_type: "기본형"/"고급형"/"실속형"
    type_: "Ⅰ"/"Ⅱ"/"Ⅲ"
    scenario:
      Ⅰ·Ⅱ용: "non_surgery"(수술 안 한 날) / "surgery"(수술한 날)
      Ⅲ용: "non_surgery_mri_1st" / "non_surgery_mri_2nd_plus" / "non_surgery_no_mri" / "surgery"
            (MRI와 수술이 같은 날 겹치면 반드시 "surgery"를 사용할 것)

    반환값에 copay_options(자기부담금 선택지, 통원/입원Ⅰ·Ⅱ·Ⅲ 공통)도 함께 포함.
    """
    if type_ in ("Ⅰ", "Ⅱ"):
        entry = LIMIT_TABLE[(coverage_type, type_)]
    elif type_ == "Ⅲ":
        entry = LIMIT_TABLE_TYPE3[coverage_type]
    else:
        raise ValueError(f"알 수 없는 유형: {type_}")

    return {
        "limit": entry[scenario],
        "rate": entry["rate"],
        "annual_total": entry["annual_total"],
        "copay_options": COPAY_OPTIONS,
    }


def calculate_payout(treatment_cost: int, copay: int, rate: float, daily_limit: int) -> int:
    """
    보험금 지급금액 산출 공식 (약관 요약서 21쪽 원문 그대로):
    = min[ (치료비 - 자기부담금) × 보상비율 , 1일당 지급한도 ]

    검증 예시(원문 그대로, 전부 일치 확인됨):
      calculate_payout(230_000, 30_000, 0.7, 150_000) == 140_000     (기본형Ⅰ 비수술)
      calculate_payout(4_000_000, 30_000, 0.7, 2_000_000) == 2_000_000  (기본형Ⅰ 수술)
      calculate_payout(4_100_000, 30_000, 0.7, 2_500_000) == 2_500_000  (기본형Ⅱ 수술)
      calculate_payout(130_000, 30_000, 0.5, 100_000) == 50_000       (실속형Ⅰ 비수술)
      calculate_payout(330_000, 30_000, 0.7, 300_000) == 210_000      (고급형Ⅲ 비수술·MRI없음)
      calculate_payout(1_030_000, 30_000, 0.7, 1_000_000) == 700_000  (고급형Ⅲ 비수술·MRI 1회차)
      calculate_payout(4_100_000, 30_000, 0.7, 2_500_000) == 2_500_000  (고급형Ⅲ MRI+수술 동시 → 수술한도)
    """
    return min(int((treatment_cost - copay) * rate), daily_limit)
'''

output_path = DATA_DIR / "limit_table.py"
output_path.write_text(limit_table_code, encoding="utf-8")
print("저장 완료:", output_path)

저장 완료: /home/kopo17/AI_programming/petmint-rag-chatbot/data/limit_table.py


In [11]:
import random
random.seed(42)
from collections import defaultdict
by_sec = defaultdict(list)
for r in records_cat:
    by_sec[r["section"]].append(r)
for sec in sorted(by_sec):
    r = random.choice(by_sec[sec])
    print(f"### {sec} | {r['section_name']}")
    print(f"조{r['article_num']} {r['paragraph_num']}항 | {r['coverage_type']} | type={r['type']} | p{r['printed_page_range']}")
    print(r["content"][:200])
    print()

### 기타_제도성_특별약관 | 보장성보험 보험료 납입유예 특별약관
조5 1항 | None | type=None | p[261, 261]
납입유예를 신청한 경우 계약자는 신청 이후 최초 도래하는 해당계
약의 납입기일부터 납입유예기간동안 보험료를 납입하지 않으며 이 경
우에도 회사는 보험료의 납입이 유예된 기간(이하 ‘납입유예기간’이
라 합니다) 동안 보험료가 정상적으로 납입된 경우와 동일하게 보장하
고, 계약자적립액 등을 계산합니다.

### 법규 | 【법규1】 개인정보 보호법
조24 1항 | None | type=None | p[218, 218]
① 개인정보처리자는 다음 각 호의 경우를 제외하고는 법령
에 따라 개인을 고유하게 구별하기 위하여 부여된 식별정보
로서 대통령령으로 정하는 정보(이하 "고유식별정보"라 한
다)를 처리할 수 없다.
  1. 정보주체에게 제15조제2항 각 호 또는 제17조제2항 각
   호의 사항을 알리고 다른 개인정보의 처리에 대한 동
   의와 별도로 동의를 받은 경우
 

### 별첨 | 특정질병 분류표(반려묘)
조None None항 | None | type=None | p[175, 179]
뒷다리근골격계질환 (관련 질병코드 30개)
- AEB003: 뒷다리의 골육종
- AEA004: 기타 근골격 계통의 양성 신생물(뒷다리)
- AEB004: 기타 근골격 계통의 악성 신생물(뒷다리)
- AEC004: 기타 근골격 계통의 악성 신생물(뒷다리) (양 성 또는 악성이 불확실한)
- NAA001: 고관절 이형성증 (좌측)
- NAA002: 고관절 이형

### 별표 | 【별표2】 장해분류표
조None None항 | None | type=None | p[207, 212]
나. 장해판정기준


 1) 신경계
   가) “신경계에 장해를 남긴 때”라 함은 뇌, 척수
    및 말초신경계 손상으로 “<붙임>일상생활 기본
       동작(ADLs) 제한 장해평가표”의 5가지 기본동작
    중 하나 이상의 동작이 제한되었을 때를 말한다.
   